# Wish ViT-B/16 real vs fake faces: dual T4 training and a tested Gradio app

This notebook fine tunes a ViT-B/16 real vs fake face detector on
[Wish RealVsFake](https://www.kaggle.com/datasets/wish096/realvsfake-81k-by-wish) using **both GPUs of
Kaggle's "GPU T4 x2" accelerator at the same time**. It then serves the trained model as a **Gradio app**
and tests that app with actual real and fake images from the dataset. No team CSVs, Git clone or code ZIP
is required: the notebook contains its own Python modules.

## Before you run
1. Settings > Accelerator > **GPU T4 x2**. A single GPU also works: set `N_GPUS = 1` and choose a new `RUN_NAME`.
2. Settings > Internet **on** (pip packages, pretrained weights, Gradio share link).
3. Attach the Wish dataset, start a fresh session and run the cells in order.

This is an **independent ViT experiment**, not a completed four model comparison. The baseline is
ImageNet 21k pretrained, ImageNet 1k fine tuned ViT-B/16, then fine tuned on Wish. No detector
performance is claimed before a real run. Test evaluation stays disabled until you explicitly freeze the experiment.

## How both GPUs are used
Training runs under `torchrun` with PyTorch DistributedDataParallel (DDP): one process per T4, each with its
own copy of the model. Every optimizer step still sees 32 images, as in the single GPU recipe, but each GPU
computes 16 of them and the gradients are averaged over NCCL while the backward pass is still running.
Learning rate, warmup, clipping, early stopping and checkpoints therefore follow the same recipe; only the
wall clock time changes. Validation is split across both GPUs and the per image results are gathered back,
so validation loss and metrics are computed over the whole validation set exactly once.

`nn.DataParallel` is deliberately not used: it drives both GPUs from one Python process, copies the model
to the second GPU on every step and overloads the first one, which rarely gets near 2x on T4s.

Two more changes keep both GPUs fed from only 4 CPU cores:
- An **exact presize cache**: train and validation images are decoded and resized to 224x224 once per
  session into local scratch space (`/tmp`, never published). Random augmentation is then applied to those
  pixels, which gives tensors identical to reading the original files, so results do not change, only speed.
- An **NCCL preflight** checks and times GPU to GPU communication before training and retries with
  `NCCL_P2P_DISABLE=1` if the direct PCIe path misbehaves, instead of hanging in the middle of training.

Expect close to, but below, 2x the single GPU throughput. The training log prints images per second and the
fraction of time spent waiting for data, so the actual speedup is measured, not assumed.

## Run it with the browser closed
Only a saved (batch) run is browser independent. Choose **Save Version > Save & Run All**, keep
**Always save output** enabled under Advanced settings, and wait until Kaggle confirms the version
is queued or running; Chrome may then be closed, quit or crash. An interactive session is not
browser independent: it ends after roughly 20 idle minutes and its unsaved working files are lost.
Quick Save does not execute anything. The first cell prints which mode is actually running.

A saved run is cancelled at the 12 hour session limit, and a cancelled or failed version publishes no
output. So the trainer stops early when the measured epoch time plus `SESSION_RESERVE_SECONDS` would
exceed the limit, never starts an epoch the remaining time cannot finish, and records a failed or hung
training run instead of raising, so completed checkpoints still reach the version output. The Gradio
section also never fails a saved run. With both GPUs, 30 epochs usually fit in one saved run; if not,
continue as below. The weekly GPU quota and the roughly 20 GB output cap remain outside the notebook's control.

## Continue after a closed browser, a crash or a stopped run
1. Open the finished version and confirm `runs/<RUN_NAME>/checkpoints/last.pt` exists in its output.
2. Start a new version and attach that output through **Add Input > Your Work > Notebook Output**.
3. Leave `RESTORE_FROM = "auto"`; it selects the attached output with the most progress for this `RUN_NAME`.
   Name the exact folder instead when several are attached.
4. Keep `RUN_NAME`, the dataset, `OUTPUT_ROOT`, `N_GPUS` and every training setting unchanged, then Save & Run All again.

Only completed epochs resume; partial epoch work is repeated. This notebook writes to a new `OUTPUT_ROOT`
and `RUN_NAME`, so it never mixes with single GPU runs. If you attach an output of the older single GPU
notebook, only its `data_audit` (the frozen splits) is reused, so both experiments share identical splits.

In [ ]:
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shutil, signal, subprocess, sys, threading, time

SEED = 42
N_GPUS = 2                 # Kaggle "GPU T4 x2": one DDP training process per GPU
EPOCHS_PER_SAVED_RUN = 30  # the time guard still pauses before the session limit; a later version resumes
os.environ["WISH_EPOCHS_PER_INVOCATION"] = str(EPOCHS_PER_SAVED_RUN)
SESSION_LIMIT_SECONDS = 12 * 3600  # Kaggle GPU notebook session limit
SESSION_RESERVE_SECONDS = 45 * 60  # kept for verification, dashboards, the Gradio self test and publishing
NOTEBOOK_START = time.perf_counter()
KAGGLE_RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Localhost")
AUDIT_WORKERS = 8          # bounded concurrent image reads, not GPU workers
MICROBATCH_PER_GPU = 16    # 16 per GPU x 2 GPUs = 32 per step, no accumulation; 8 if a T4 runs out of memory (new RUN_NAME)
EFFECTIVE_BATCH = 32       # images per optimizer step, the same recipe as the single GPU notebook
NUM_WORKERS_PER_GPU = 2    # DataLoader processes per GPU; the T4 x2 machine has 4 CPU cores
PRESIZE_CACHE = True       # exact 224x224 decode cache in local scratch space: speed only, results unchanged
PRESIZE_CACHE_DIR = Path("/tmp/wish_presize_cache")
EPOCHS = 30
RUN_NAME = "vit_wish_s42_ddp2_v1"
RUN_FULL_TRAINING = True
READY_FOR_FINAL_EVALUATION = False
RUN_GRADIO_APP = True             # final section: launch the Gradio app and self test it on real and fake images
GRADIO_SELF_TEST_PER_SOURCE = 12  # dataset images per source (FFHQ, CelebA, StyleGAN; Stable Diffusion once sealed)
OWN_TEST_IMAGES = None     # optional folder of your own images, e.g. an attached dataset with real/ and fake/ subfolders
DATASET_INPUT = None       # optional: exact folder containing the attached Wish dataset
KAGGLE_VERSION = None      # optional known integer; None uses attached data and records content hashes
OUTPUT_ROOT = Path("/kaggle/working/wish_vit_ddp_v4")
RESTORE_FROM = "auto"      # "auto" resumes an attached earlier output; a path forces one; None starts fresh
KAGGLE_INPUTS = Path("/kaggle/input")

assert N_GPUS >= 1 and 1 <= EPOCHS <= 30
assert MICROBATCH_PER_GPU > 0 and EFFECTIVE_BATCH % (MICROBATCH_PER_GPU * N_GPUS) == 0, \
    "EFFECTIVE_BATCH must be a multiple of MICROBATCH_PER_GPU x N_GPUS"
assert Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")


def saved_outputs(inputs=KAGGLE_INPUTS):
    """Earlier outputs of this notebook (or of the single GPU version) attached as inputs."""
    roots = []
    for depth in range(1, 5):
        for audit in sorted(inputs.glob("/".join(["*"] * depth) + "/data_audit/audit.json")):
            if audit.parent.parent not in roots:
                roots.append(audit.parent.parent)
    return roots


def progress_of(root, run_name=RUN_NAME):
    run_dir = root / "runs" / run_name
    rows = run_dir / "training_history.csv"
    epochs = max(len([line for line in rows.read_text().splitlines() if line.strip()]) - 1, 0) if rows.is_file() else 0
    return (run_dir / "training_complete.json").is_file(), (run_dir / "checkpoints/last.pt").is_file(), epochs


def find_restore_root():
    """Furthest progress on this RUN_NAME wins; an output without it still supplies the frozen splits."""
    found = sorted((progress_of(root), str(root)) for root in saved_outputs())
    return found[-1][1] if found else None


if RESTORE_FROM == "auto":
    RESTORE_FROM = find_restore_root()
    print("Auto resume source:", RESTORE_FROM or "none attached; this starts a new run")
if RESTORE_FROM and not OUTPUT_ROOT.exists():
    saved = Path(RESTORE_FROM)
    assert (saved / "data_audit/audit.json").is_file(), "RESTORE_FROM must contain the full saved output folder"
    OUTPUT_ROOT.mkdir(parents=True)
    shutil.copytree(saved / "data_audit", OUTPUT_ROOT / "data_audit")  # frozen splits, reused unchanged
    if (saved / "runs" / RUN_NAME).is_dir():  # the same experiment: continue it
        for name in (RUN_NAME, f"{RUN_NAME}_smoke"):
            if (saved / "runs" / name).is_dir():
                shutil.copytree(saved / "runs" / name, OUTPUT_ROOT / "runs" / name)
        for name in (f"{RUN_NAME}.json", f"{RUN_NAME}_smoke.json"):
            if (saved / name).is_file():
                shutil.copy2(saved / name, OUTPUT_ROOT / name)
        if (saved / "sealed_evaluation").is_dir():
            shutil.copytree(saved / "sealed_evaluation", OUTPUT_ROOT / "sealed_evaluation")
        print(f"Continuing run {RUN_NAME}: {progress_of(saved)[2]} epoch(s) already recorded.")
    else:
        print(f"No run named {RUN_NAME} there: reusing only its frozen data splits; training starts fresh.")
elif RESTORE_FROM:
    print("Using existing output directory; not overwriting it with restored files.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def run(*args, timeout=None, env=None):
    """Run `python *args`, stream its output into this cell, and on a timeout or an interrupt stop the
    whole process tree (torchrun, every GPU worker and their data loaders), so nothing keeps a GPU busy."""
    command = [sys.executable, *map(str, args)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                               errors="replace", env=env, start_new_session=True)

    def pump():
        for line in process.stdout:
            print(line, end="", flush=True)

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    try:
        returncode = process.wait(timeout=timeout)
    except BaseException:
        for stop in (signal.SIGTERM, signal.SIGKILL):
            try:
                os.killpg(process.pid, stop)
                process.wait(timeout=30)
                break
            except ProcessLookupError:
                break
            except subprocess.TimeoutExpired:
                continue
        raise
    finally:
        reader.join(timeout=10)
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


# Retain Kaggle's CUDA enabled torch/torchvision pair. Install only missing helpers.
packages = {"timm": "timm==1.0.30", "pandas": "pandas", "numpy": "numpy",
            "PIL": "Pillow", "yaml": "PyYAML", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "kagglehub": "kagglehub"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    run("-m", "pip", "install", *missing)
import torch, torchvision, timm, pandas as pd
assert torch.cuda.is_available(), "Select Accelerator > GPU T4 x2, then start a fresh session."
assert hasattr(torch.amp, "GradScaler"), "This notebook needs PyTorch 2.3+ with torch.amp.GradScaler."
GPU_COUNT = torch.cuda.device_count()
for index in range(GPU_COUNT):
    print(f"GPU {index}:", torch.cuda.get_device_name(index))
assert GPU_COUNT >= N_GPUS, (f"{GPU_COUNT} GPU(s) visible but N_GPUS = {N_GPUS}. Select Accelerator > GPU T4 x2, "
                             "or set N_GPUS = 1 and a new RUN_NAME.")
print(f"Training uses {N_GPUS} GPU(s) at once: {MICROBATCH_PER_GPU} images per GPU, "
      f"{EFFECTIVE_BATCH} per optimizer step. CPU cores: {os.cpu_count()}")
print("torch:", torch.__version__, "torchvision:", torchvision.__version__, "timm:", timm.__version__)
print("Outputs:", OUTPUT_ROOT)
print("Kaggle run type:", KAGGLE_RUN_TYPE)
if KAGGLE_RUN_TYPE == "Batch":
    print("Saved (batch) run: execution continues after the browser is closed or crashes.")
else:
    print("INTERACTIVE SESSION. Closing or crashing the browser ends it after the idle timeout")
    print("and its working files are lost. For a browser independent run choose Save Version >")
    print("Save & Run All, keep Always save output enabled, and wait for the queued/running")
    print("confirmation before closing Chrome. Quick Save does not execute the notebook.")

## Included runtime
The next cell installs the notebook's readable source modules into its own output directory, including
`distributed.py` (process group helpers), `presize_cache.py`, `launch_train.py` (the `torchrun` entry point)
and `nccl_check.py`. Do not run it in a session that already imported modules from an older notebook; restart first.

In [ ]:
SOURCE_FILES = {
    'models/vit/manifest.py': '"""Validate Member A\'s Wish assignments without generating replacement splits."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nfrom PIL import Image, ImageOps\n\nDATASET = "wish096/realvsfake-81k-by-wish"\nSPLITS = ("train", "val", "test", "cross_gen")\nSOURCES = {"RFF": (0, "FFHQ"), "RCA": (0, "CelebA"),\n           "FSG": (1, "StyleGAN"), "FSD": (1, "StableDiffusion"),\n           "AI": (1, "AiGenImage")}\nFIELDS = ("filepath", "label", "source", "split", "identity", "sha256", "pixel_sha256", "dhash")\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef source_from_name(path):\n    match = re.fullmatch(r"(RFF|RCA|FSG|FSD|AI)\\s*\\(\\d+\\)\\.(?:jpg|jpeg|png|webp)",\n                         Path(path).name, flags=re.IGNORECASE)\n    if not match:\n        raise ValueError(f"Unrecognized Wish filename: {path}; retain original filenames.")\n    return SOURCES[match.group(1).upper()]\n\n\ndef image_path(root, relative):\n    path = Path(relative)\n    if path.is_absolute() or ".." in path.parts:\n        raise ValueError(f"filepath must be relative to data root: {relative}")\n    resolved = (Path(root).resolve() / path).resolve()\n    if not resolved.is_relative_to(Path(root).resolve()):\n        raise ValueError(f"Image escapes data root: {relative}")\n    return resolved\n\n\ndef read_manifest(path):\n    with open(path, newline="", encoding="utf-8-sig") as handle:\n        reader = csv.DictReader(handle)\n        required = {"filepath", "label", "source", "split"}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError(f"Manifest requires columns {sorted(required)}")\n        rows = list(reader)\n    if not rows:\n        raise ValueError("Manifest is empty.")\n    return rows\n\n\ndef combine_split_csvs(split_files, output, strip_prefix=""):\n    """Convert A\'s existing split files to the shared schema; never reassign a row."""\n    if not split_files or not set(split_files).issubset(set(SPLITS) | {"excluded"}):\n        raise ValueError("CSV keys must be train, val, test, cross_gen (and optional excluded).")\n    rows = []\n    prefix = strip_prefix.replace("\\\\", "/")\n    for split, path in split_files.items():\n        with open(path, newline="", encoding="utf-8-sig") as handle:\n            reader = csv.DictReader(handle)\n            if not {"filepath", "label"}.issubset(reader.fieldnames or []):\n                raise ValueError(f"{path} requires filepath and label columns.")\n            for row in reader:\n                if row.get("split") and row["split"] != split:\n                    raise ValueError(f"CSV filename assignment conflicts with row split: {path}")\n                relative = row["filepath"].replace("\\\\", "/")\n                if prefix:\n                    if not relative.startswith(prefix):\n                        raise ValueError(f"Explicit strip prefix does not match: {relative}")\n                    relative = relative[len(prefix):]\n                # This changes only path notation; source/label are cross-checked.\n                if Path(relative).is_absolute() or ".." in Path(relative).parts:\n                    raise ValueError("CSV paths must be relative; set the exact MEMBER_A_PATH_PREFIX if needed.")\n                _, known_source = source_from_name(relative)\n                rows.append({"filepath": relative, "label": row["label"],\n                             "source": row.get("source") or known_source,\n                             "split": split, "identity": row.get("identity", "")})\n    # Missing cross-generator controls or SD in train/val must be corrected by A.\n    rows = validate_rows(rows)\n    output = Path(output)\n    comparable = lambda entries: [{k: str(row.get(k, "")) for k in FIELDS[:5]} for row in entries]\n    if output.exists():\n        if comparable(read_manifest(output)) != comparable(rows):\n            raise ValueError("Combined manifest already exists with different assignments; choose a new output directory.")\n        return output\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open("w", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=FIELDS[:5])\n        writer.writeheader()\n        writer.writerows(rows)\n    return output\n\n\ndef validate_rows(rows, require_all=True):\n    seen, identities, counts = set(), {}, Counter()\n    result = []\n    for original in rows:\n        row = dict(original)\n        row["filepath"] = row["filepath"].replace("\\\\", "/")\n        if row["filepath"] in seen:\n            raise ValueError(f"Repeated filepath / split overlap: {row[\'filepath\']}")\n        seen.add(row["filepath"])\n        label, source = source_from_name(row["filepath"])\n        if str(row["label"]) != str(label) or row["source"] != source:\n            raise ValueError(f"Filename/label/source conflict: {row[\'filepath\']}")\n        split = row["split"]\n        if source == "AiGenImage":\n            if split != "excluded":\n                raise ValueError("AiGenImage must be explicitly assigned to excluded by Member A.")\n        elif split not in SPLITS:\n            raise ValueError(f"Unknown split: {split}")\n        elif source == "StableDiffusion" and split != "cross_gen":\n            raise ValueError("StableDiffusion leakage: only cross_gen is permitted.")\n        elif source == "StyleGAN" and split == "cross_gen":\n            raise ValueError("StyleGAN cannot appear in the held-out cross_gen set.")\n        identity = row.get("identity", "").strip()\n        if identity and split != "excluded":\n            key = (source, identity)\n            if key in identities and identities[key] != split:\n                raise ValueError(f"Identity overlaps splits: {key}")\n            identities[key] = split\n        row["label"] = label\n        row["identity"] = identity\n        counts[(split, label)] += 1\n        result.append(row)\n    if require_all:\n        for split in SPLITS:\n            for label in (0, 1):\n                if not counts[(split, label)]:\n                    raise ValueError(f"{split} requires both classes; missing label {label}.")\n    return result\n\n\ndef audit_manifest(manifest, root, output, dataset_version, near_distance=4):\n    if not str(dataset_version).strip():\n        raise ValueError("Record the downloaded Kaggle dataset version.")\n    if not 0 <= near_distance <= 8:\n        raise ValueError("near_distance must be between 0 and 8.")\n    rows = validate_rows(read_manifest(manifest))\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError(f"Refusing to overwrite audit: {output}")\n    file_seen, pixel_seen, buckets, near = {}, {}, {}, []\n    counts, dimensions = Counter(), Counter()\n    # A d-bit neighbor must share one of d+1 hash pieces. This avoids a\n    # quadratic scan while retaining every dHash candidate within distance d.\n    pieces = near_distance + 1\n    for row in rows:\n        path = image_path(root, row["filepath"])\n        digest = sha256_file(path)\n        with Image.open(path) as image:\n            image.load()\n            rgb = ImageOps.exif_transpose(image).convert("RGB")\n            dimensions[f"{rgb.width}x{rgb.height}"] += 1\n            pixel_hash = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()\n            small = list(rgb.convert("L").resize((9, 8)).tobytes())\n            value = sum(int(small[y * 9 + x] > small[y * 9 + x + 1]) << (y * 8 + x)\n                        for y in range(8) for x in range(8))\n        row.update(sha256=digest, pixel_sha256=pixel_hash, dhash=f"{value:016x}")\n        counts[f"{row[\'split\']}:{row[\'source\']}:{row[\'label\']}"] += 1\n        if row["split"] == "excluded":\n            continue\n        for index, fingerprint in ((file_seen, digest), (pixel_seen, pixel_hash)):\n            if fingerprint in index:\n                raise ValueError(f"Duplicate image content: {row[\'filepath\']} and {index[fingerprint]}")\n            index[fingerprint] = row["filepath"]\n        candidates = {}\n        keys = []\n        for piece in range(pieces):\n            start, end = 64 * piece // pieces, 64 * (piece + 1) // pieces\n            key = (piece, (value >> start) & ((1 << (end - start)) - 1))\n            keys.append(key)\n            for previous in buckets.get(key, []):\n                candidates[previous[0]] = previous\n        for previous_path, previous_split, previous_hash in candidates.values():\n            distance = (value ^ previous_hash).bit_count()\n            if previous_split != row["split"] and distance <= near_distance:\n                near.append({"filepath_a": previous_path, "filepath_b": row["filepath"],\n                             "distance": distance})\n        for key in keys:\n            buckets.setdefault(key, []).append((row["filepath"], row["split"], value))\n    output.mkdir(parents=True)\n    with open(output / "manifest.csv", "w", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=FIELDS)\n        writer.writeheader()\n        writer.writerows({key: row.get(key, "") for key in FIELDS} for row in rows)\n    with open(output / "near_duplicates.csv", "w", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=("filepath_a", "filepath_b", "distance"))\n        writer.writeheader()\n        writer.writerows(near)\n    report = {"dataset": DATASET, "dataset_version": str(dataset_version),\n              "manifest_sha256": sha256_file(output / "manifest.csv"),\n              "input_manifest_sha256": sha256_file(manifest), "counts": dict(counts),\n              "dimensions": dict(dimensions), "near_duplicate_pairs": len(near),\n              "near_distance": near_distance,\n              "identity_metadata_available": any(row["identity"] for row in rows),\n              "status": "needs_near_duplicate_review" if near else "passed"}\n    (output / "audit.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\ndef load_audited_manifest(manifest, root, review=None, verify_splits=SPLITS):\n    manifest = Path(manifest)\n    rows = validate_rows(read_manifest(manifest))\n    audit = json.loads(manifest.with_name("audit.json").read_text())\n    if audit["dataset"] != DATASET or audit["manifest_sha256"] != sha256_file(manifest):\n        raise ValueError("Audit does not match this Wish manifest. Re-audit through Member A.")\n    if audit["near_duplicate_pairs"]:\n        if not review:\n            raise ValueError("Review near_duplicates.csv and provide a signed review JSON before training.")\n        decision = json.loads(Path(review).read_text())\n        if (decision.get("manifest_sha256") != audit["manifest_sha256"]\n                or decision.get("decision") != "false_positives_only"\n                or not decision.get("reviewer") or not decision.get("rationale")):\n            raise ValueError("Review must identify reviewer, rationale and audited manifest.")\n    for row in rows:\n        if row["split"] in verify_splits:\n            if row.get("sha256") != sha256_file(image_path(root, row["filepath"])):\n                raise ValueError(f"Image changed since audit: {row[\'filepath\']}")\n    return rows, audit\n',
    'models/vit/runtime.py': '"""Shared inference contract for training, evaluation and the Member C demo."""\nfrom __future__ import annotations\n\nimport copy\nimport importlib\nimport inspect\nimport io\nimport json\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport yaml\nfrom PIL import Image, ImageOps\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms as T\n\nfrom models.vit.manifest import image_path\n\n\ndef load_config(path):\n    with open(path) as handle:\n        cfg = yaml.safe_load(handle)\n    if not isinstance(cfg, dict):\n        raise ValueError("Configuration must be a mapping.")\n    return cfg\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef device_for(name="auto"):\n    if name != "auto":\n        return torch.device(name)\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef probabilities(logits):\n    if logits.ndim == 1:\n        return logits.sigmoid()\n    if logits.ndim == 2 and logits.shape[1] == 1:\n        return logits[:, 0].sigmoid()\n    if logits.ndim == 2 and logits.shape[1] == 2:\n        return logits.softmax(dim=1)[:, 1]\n    raise ValueError(f"Expected [B], [B,1] or [B,2] logits, got {tuple(logits.shape)}")\n\n\ndef binary_loss(logits, labels):\n    if logits.ndim == 1 or (logits.ndim == 2 and logits.shape[1] == 1):\n        return torch.nn.functional.binary_cross_entropy_with_logits(logits.reshape(-1), labels.float())\n    if logits.ndim == 2 and logits.shape[1] == 2:\n        return torch.nn.functional.cross_entropy(logits, labels.long())\n    raise ValueError("Unsupported classifier output.")\n\n\ndef per_sample_loss(logits, labels):\n    """Unreduced loss, so metrics can be computed exactly once per image across GPUs."""\n    if logits.ndim == 1 or (logits.ndim == 2 and logits.shape[1] == 1):\n        return torch.nn.functional.binary_cross_entropy_with_logits(\n            logits.reshape(-1), labels.float(), reduction="none")\n    if logits.ndim == 2 and logits.shape[1] == 2:\n        return torch.nn.functional.cross_entropy(logits, labels.long(), reduction="none")\n    raise ValueError("Unsupported classifier output.")\n\n\ndef build_model(cfg, pretrained=None):\n    cfg = copy.deepcopy(cfg)\n    module = importlib.import_module(cfg["module"])\n    if pretrained is not None:\n        cfg.setdefault("model", {})["pretrained"] = pretrained\n        if not pretrained:\n            cfg["model"]["weights"] = False\n    if hasattr(module, "build_model"):\n        return module.build_model(cfg)\n    if hasattr(module, "get_model"):\n        kwargs = {"num_classes": cfg.get("model", {}).get("num_classes", 2)}\n        if "pretrained" in inspect.signature(module.get_model).parameters:\n            kwargs["pretrained"] = cfg.get("model", {}).get("pretrained", True)\n        return module.get_model(**kwargs)\n    raise ValueError(f"{cfg[\'module\']} exposes neither build_model nor get_model.")\n\n\ndef preprocessing_for(model, cfg):\n    backbone = getattr(model, "backbone", model)\n    if hasattr(backbone, "pretrained_cfg"):\n        from timm.data import resolve_model_data_config\n        data = resolve_model_data_config(backbone)\n        mean, std = data["mean"], data["std"]\n        interpolation = data.get("interpolation", "bicubic")\n    else:\n        data = cfg.get("preprocessing", {})\n        mean = data.get("mean", [0.485, 0.456, 0.406])\n        std = data.get("std", [0.229, 0.224, 0.225])\n        interpolation = data.get("interpolation", "bilinear")\n    return {"image_size": 224, "mean": list(mean), "std": list(std),\n            "interpolation": interpolation, "color": "RGB", "resize": "square",\n            "crop": "already_cropped_wish_faces"}\n\n\nclass JPEGRecompress:\n    def __call__(self, image):\n        buffer = io.BytesIO()\n        image.save(buffer, format="JPEG", quality=random.randint(70, 100))\n        buffer.seek(0)\n        with Image.open(buffer) as decoded:\n            return decoded.convert("RGB")\n\n\ndef transform_for(metadata, train=False, resized=False):\n    """resized=True skips only the deterministic first Resize (input comes from the exact presize cache)."""\n    interpolation = T.InterpolationMode(metadata["interpolation"])\n    ops = [] if resized else [T.Resize((metadata["image_size"], metadata["image_size"]), interpolation=interpolation)]\n    if train:\n        ops += [T.RandomHorizontalFlip(), T.RandomRotation(10),\n                T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),\n                T.RandomApply([JPEGRecompress()], p=0.3),\n                T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.2)]\n    return T.Compose(ops + [T.ToTensor(), T.Normalize(metadata["mean"], metadata["std"])])\n\n\nclass ManifestDataset(Dataset):\n    def __init__(self, rows, root, split, metadata, train=False, limit=None, cache=None):\n        self.rows = [row for row in rows if row["split"] == split]\n        if limit:\n            self.rows = [row for label in (0, 1)\n                         for row in [r for r in self.rows if r["label"] == label][:max(1, limit // 2)]]\n        if not self.rows:\n            raise ValueError(f"Empty dataset: {split}")\n        self.root = root\n        self.transform = transform_for(metadata, train)\n        self.cache = cache\n        self.cached_transform = transform_for(metadata, train, resized=True) if cache is not None else None\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        resized = self.cache.get(row["filepath"]) if self.cache is not None else None\n        if resized is not None:\n            return self.cached_transform(resized), int(row["label"]), index\n        with Image.open(image_path(self.root, row["filepath"])) as image:\n            tensor = self.transform(ImageOps.exif_transpose(image).convert("RGB"))\n        return tensor, int(row["label"]), index\n\n\ndef load_checkpoint(cfg, path, device):\n    model = build_model(cfg, pretrained=False).to(device)\n    checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n    state = checkpoint.get("model_state", checkpoint.get("state_dict", checkpoint))\n    model.load_state_dict(state, strict=True)\n    metadata = checkpoint.get("metadata")\n    if metadata is None:\n        sidecar = Path(str(path) + ".json")\n        if not sidecar.exists():\n            raise ValueError("Legacy checkpoint requires a verified <checkpoint>.json metadata sidecar.")\n        metadata = json.loads(sidecar.read_text())\n    if metadata.get("model_config") != cfg["model"] or metadata.get("module") != cfg["module"]:\n        raise ValueError("Checkpoint metadata does not match the selected model configuration.")\n    if metadata.get("label_mapping") != {"real": 0, "fake": 1}:\n        raise ValueError("Checkpoint must declare real=0 and fake=1.")\n    return model.eval(), metadata\n\n\n@torch.inference_mode()\ndef predict_image(model, image, metadata, device):\n    """P(fake) for one PIL image; accepts checkpoint metadata or its bare preprocessing block."""\n    preprocessing = metadata.get("preprocessing", metadata)\n    tensor = transform_for(preprocessing)(ImageOps.exif_transpose(image).convert("RGB"))\n    return float(probabilities(model(tensor.unsqueeze(0).to(device)))[0].cpu())\n',
    'models/vit/model.py': '"""\nModel 4: ViT-B/16 (Transfer Learning) + optional frequency-hybrid branch.\nOwner: Member C\n\nThis is the ONLY place the network architecture is defined. Hyperparameters\n(learning rate, epochs, whether the frequency branch is on, etc.) all come\nfrom configs/vit.yaml — do not hardcode them here.\n"""\n\nimport torch\nimport torch.nn as nn\n\ntry:\n    import timm\nexcept ImportError as e:\n    raise ImportError(\n        "timm is required for the ViT backbone. Add it to requirements.txt "\n        "and `pip install timm`."\n    ) from e\n\n\nclass FrequencyBranch(nn.Module):\n    """\n    Lightweight frequency-domain branch (novelty add-on).\n\n    Takes the 2D FFT magnitude spectrum of the input image and runs it through\n    a small CNN. GAN / diffusion upsampling artifacts often show up more\n    clearly in frequency space than in pixel space, so fusing this signal\n    with the ViT\'s spatial features can help cross-generator generalization.\n    """\n\n    def __init__(self, in_channels: int = 3, conv_channels=(16, 32, 64), fusion_dim: int = 128):\n        super().__init__()\n        layers = []\n        prev = in_channels\n        for ch in conv_channels:\n            layers += [\n                nn.Conv2d(prev, ch, kernel_size=3, stride=2, padding=1),\n                nn.BatchNorm2d(ch),\n                nn.ReLU(inplace=True),\n            ]\n            prev = ch\n        self.conv = nn.Sequential(*layers)\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.proj = nn.Linear(prev, fusion_dim)\n\n    @staticmethod\n    def to_fft_magnitude(x: torch.Tensor) -> torch.Tensor:\n        """Convert a batch of RGB images (B, C, H, W) to log-magnitude FFT spectra."""\n        # CUDA half-precision FFT does not support the non-power-of-two size 224.\n        with torch.autocast(device_type=x.device.type, enabled=False):\n            fft = torch.fft.fft2(x.float(), norm="ortho")\n        fft_shifted = torch.fft.fftshift(fft, dim=(-2, -1))\n        magnitude = torch.log(torch.abs(fft_shifted) + 1e-8)\n        return magnitude\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        freq = self.to_fft_magnitude(x)\n        feat = self.conv(freq)\n        feat = self.pool(feat).flatten(1)\n        return self.proj(feat)\n\n\nclass ViTDeepfakeDetector(nn.Module):\n    """\n    ViT-B/16 backbone (pretrained, via timm) with a binary classification head.\n    If `use_frequency_hybrid` is enabled, the FFT branch\'s features are\n    concatenated with the ViT\'s pooled output before the final linear layer.\n    """\n\n    def __init__(\n        self,\n        backbone: str = "vit_base_patch16_224.augreg_in21k_ft_in1k",\n        pretrained: bool = True,\n        num_classes: int = 1,\n        dropout: float = 0.1,\n        use_frequency_hybrid: bool = False,\n        freq_branch_cfg: dict | None = None,\n    ):\n        super().__init__()\n        self.use_frequency_hybrid = use_frequency_hybrid\n\n        # num_classes=0 -> timm returns pooled features instead of its own head\n        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)\n        vit_feat_dim = self.backbone.num_features\n\n        if use_frequency_hybrid:\n            freq_branch_cfg = freq_branch_cfg or {}\n            self.freq_branch = FrequencyBranch(\n                in_channels=3,\n                conv_channels=freq_branch_cfg.get("conv_channels", [16, 32, 64]),\n                fusion_dim=freq_branch_cfg.get("fusion_dim", 128),\n            )\n            head_in_dim = vit_feat_dim + freq_branch_cfg.get("fusion_dim", 128)\n        else:\n            self.freq_branch = None\n            head_in_dim = vit_feat_dim\n\n        self.dropout = nn.Dropout(dropout)\n        self.classifier = nn.Linear(head_in_dim, num_classes)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        vit_feat = self.backbone(x)  # (B, vit_feat_dim)\n\n        if self.use_frequency_hybrid:\n            freq_feat = self.freq_branch(x)  # (B, fusion_dim)\n            fused = torch.cat([vit_feat, freq_feat], dim=1)\n        else:\n            fused = vit_feat\n\n        fused = self.dropout(fused)\n        logits = self.classifier(fused)  # (B, num_classes)\n        return logits.squeeze(-1) if logits.shape[-1] == 1 else logits\n\n\ndef build_model(cfg: dict) -> nn.Module:\n    """Factory used by train.py — reads the `model:` section of configs/vit.yaml."""\n    m_cfg = cfg["model"]\n    return ViTDeepfakeDetector(\n        backbone=m_cfg.get("backbone", "vit_base_patch16_224.augreg_in21k_ft_in1k"),\n        pretrained=m_cfg.get("pretrained", True),\n        num_classes=m_cfg.get("num_classes", 1),\n        dropout=m_cfg.get("dropout", 0.1),\n        use_frequency_hybrid=m_cfg.get("use_frequency_hybrid", False),\n        freq_branch_cfg=m_cfg.get("freq_branch", {}),\n    )\n',
    'models/vit/reporting.py': 'from models.vit.quality_report import compute_metrics, make_figures, learning_curves\n',
    'models/vit/train.py': '"""Manifest-based training on one device or several GPUs (DDP via torchrun).\n\nThis command never evaluates either test set. With WORLD_SIZE > 1 each GPU trains\non a disjoint shard; gradients are averaged by DistributedDataParallel, so the\neffective batch is per-GPU microbatch x GPUs x accumulation, exactly as configured.\n"""\nfrom __future__ import annotations\nimport os\nimport argparse\nimport copy\nimport importlib.metadata\nimport json\nimport math\nimport platform\nimport random\nimport subprocess\nimport time\nfrom contextlib import nullcontext\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn.parallel import DistributedDataParallel\nfrom torch.utils.data import DataLoader, Sampler\nfrom torch.utils.data.distributed import DistributedSampler\nfrom models.vit import distributed as dist_utils\nfrom models.vit.manifest import load_audited_manifest\nfrom models.vit.presize_cache import open_cache\nfrom models.vit.reporting import compute_metrics, learning_curves\nfrom models.vit.runtime import ManifestDataset, binary_loss, build_model, load_config, per_sample_loss, preprocessing_for, probabilities, set_seed\n\n\nclass ShardSampler(Sampler):\n    """Disjoint, unpadded validation shards: every validation image is scored exactly once."""\n\n    def __init__(self, size, rank, world):\n        self.indices = list(range(rank, size, world))\n\n    def __iter__(self):\n        return iter(self.indices)\n\n    def __len__(self):\n        return len(self.indices)\n\n\ndef unwrap(model):\n    return model.module if isinstance(model, DistributedDataParallel) else model\n\n\ndef wrap(model, ctx):\n    """DDP after staged unfreezing; rewrapped whenever the trainable set changes."""\n    if ctx.world == 1:\n        return model\n    ids = [ctx.device.index] if ctx.device.type == \'cuda\' else None\n    return DistributedDataParallel(model, device_ids=ids, output_device=ids[0] if ids else None)\n\ndef phase_parameters(model, cfg, warmup):\n    """Keep teammates\' heads/architectures; only apply their staged unfreezing."""\n    for parameter in model.parameters():\n        parameter.requires_grad = True\n    if not cfg[\'train\'].get(\'warmup_epochs\', 0):\n        return\n    for parameter in model.parameters():\n        parameter.requires_grad = False\n    backbone = getattr(model, \'backbone\', model)\n    head = getattr(model, \'head\', None)\n    if head is None:\n        head = getattr(backbone, \'fc\', getattr(backbone, \'classifier\', None))\n    if head is None:\n        raise ValueError(\'Cannot identify classifier for staged training.\')\n    for parameter in head.parameters():\n        parameter.requires_grad = True\n    if not warmup:\n        if hasattr(backbone, \'layer4\'):\n            blocks = [backbone.layer3, backbone.layer4]\n        elif hasattr(backbone, \'blocks\'):\n            blocks = list(backbone.blocks.children())[-2:]\n        elif hasattr(backbone, \'features\'):\n            blocks = list(backbone.features.children())[-2:]\n        else:\n            raise ValueError(\'Cannot identify backbone blocks for fine-tuning.\')\n        for block in blocks:\n            for parameter in block.parameters():\n                parameter.requires_grad = True\n\ndef run_epoch(model, loader, device, ctx, optimizer=None, scheduler=None, scaler=None, accumulation=1, grad_clip=1.0, amp=False, log_every=0):\n    """One pass over this rank\'s shard; losses/scores are gathered so every rank sees whole-dataset values.\n\n    Per-step host synchronisation is limited to the GradScaler check, so the CPU can queue the\n    next step while the GPU is still busy. Nonfinite losses are checked at log points and at the end.\n    """\n    training = optimizer is not None\n    model.train(training)\n    network = model if training else unwrap(model)  # evaluation never enters a DDP collective\n    if training:\n        for module in unwrap(model).modules():\n            parameters = list(module.parameters(recurse=False))\n            if isinstance(module, torch.nn.modules.batchnorm._BatchNorm) and parameters and (not any((p.requires_grad for p in parameters))):\n                module.eval()\n        optimizer.zero_grad(set_to_none=True)\n    steps, local_samples = (len(loader), len(loader.sampler))\n    losses, scores, labels_out, indices_out = ([], [], [], [])\n    started = fetched = time.perf_counter()\n    waiting = 0.0\n    with torch.set_grad_enabled(training):\n        for step, (images, labels, indices) in enumerate(loader):\n            waiting += time.perf_counter() - fetched\n            images, labels = (images.to(device, non_blocking=True), labels.to(device, non_blocking=True))\n            boundary = (step + 1) % accumulation == 0 or step + 1 == steps\n            sync = network.no_sync() if training and not boundary and isinstance(network, DistributedDataParallel) else nullcontext()\n            with sync:\n                with torch.autocast(device_type=device.type, enabled=amp):\n                    logits = network(images)\n                    loss = binary_loss(logits, labels)\n                if training:\n                    start = step // accumulation * accumulation * loader.batch_size\n                    window_samples = min(accumulation * loader.batch_size, local_samples - start)\n                    scaler.scale(loss * len(labels) / window_samples).backward()\n            if training and boundary:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(unwrap(model).parameters(), grad_clip)\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scheduler is not None and scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            with torch.no_grad():\n                detached = logits.detach().float()\n                losses.append(per_sample_loss(detached, labels))\n                scores.append(probabilities(detached))\n            labels_out.append(labels.detach())\n            indices_out.append(indices)\n            if log_every and ((step + 1) % log_every == 0 or step + 1 == steps):\n                seen = torch.cat(losses)\n                if not torch.isfinite(seen).all():\n                    raise ValueError(\'Nonfinite loss; refusing to save a misleading result.\')\n                if ctx.main:\n                    print(f"{(\'train\' if training else \'val\')} batch={step + 1}/{steps} loss={float(seen.mean()):.4f} (rank 0 shard)", flush=True)\n            fetched = time.perf_counter()\n    seconds = time.perf_counter() - started\n    local = {\'index\': torch.cat(indices_out).tolist() if indices_out else [],\n             \'score\': torch.cat(scores).cpu().tolist() if scores else [],\n             \'label\': torch.cat(labels_out).cpu().tolist() if labels_out else [],\n             \'loss\': torch.cat(losses).cpu().tolist() if losses else [],\n             \'seconds\': seconds, \'waiting\': waiting}\n    merged = {}\n    parts = dist_utils.all_gather_object(ctx, local)\n    for part in parts:\n        for index, score, label, value in zip(part[\'index\'], part[\'score\'], part[\'label\'], part[\'loss\']):\n            merged.setdefault(index, (score, label, value))  # drops DistributedSampler padding repeats\n    if not merged:\n        raise ValueError(\'Empty loader.\')\n    order = sorted(merged)\n    values = np.asarray([merged[i][2] for i in order], dtype=np.float64)\n    if not np.isfinite(values).all():\n        raise ValueError(\'Nonfinite loss; refusing to save a misleading result.\')\n    metrics = compute_metrics([merged[i][1] for i in order], [merged[i][0] for i in order]) if ctx.main else None\n    slowest = max((part[\'seconds\'] for part in parts))\n    stats = {\'images\': len(order), \'images_per_second\': len(order) / max(slowest, 1e-09),\n             \'data_wait_fraction\': max((part[\'waiting\'] / max(part[\'seconds\'], 1e-09) for part in parts))}\n    return (float(values.mean()), metrics, stats)\n\ndef rng_state(generator, ctx=None):\n    np_state = np.random.get_state()\n    state = {\'python\': random.getstate(), \'numpy\': [np_state[0], np_state[1].tolist(), *np_state[2:]], \'torch\': torch.get_rng_state(), \'loader\': generator.get_state()}\n    if ctx is not None and ctx.device.type == \'cuda\':\n        state[\'cuda_device\'] = torch.cuda.get_rng_state(ctx.device)  # this rank\'s GPU only\n    elif ctx is None and torch.cuda.is_available():\n        state[\'cuda\'] = torch.cuda.get_rng_state_all()\n    if torch.backends.mps.is_available():\n        state[\'mps\'] = torch.mps.get_rng_state()\n    return state\n\ndef restore_rng(state, generator, ctx=None):\n    if isinstance(state, list):\n        if ctx is None or len(state) != ctx.world:\n            raise ValueError(\'Saved RNG states do not match the number of training processes.\')\n        state = state[ctx.rank]\n    random.setstate(state[\'python\'])\n    n = state[\'numpy\']\n    np.random.set_state((n[0], np.asarray(n[1], dtype=\'uint32\'), *n[2:]))\n    torch.set_rng_state(state[\'torch\'])\n    generator.set_state(state[\'loader\'])\n    if \'cuda_device\' in state:\n        torch.cuda.set_rng_state(state[\'cuda_device\'], ctx.device)\n    if \'cuda\' in state:\n        torch.cuda.set_rng_state_all(state[\'cuda\'])\n    if \'mps\' in state:\n        torch.mps.set_rng_state(state[\'mps\'])\n\ndef train(cfg, resume=None, ctx=None):\n    invocation_start = time.perf_counter()\n    cfg = copy.deepcopy(cfg)\n    ctx = ctx or dist_utils.setup(cfg.get(\'device\', \'auto\'))\n    t = cfg[\'train\']\n    if not 1 <= t[\'epochs\'] <= 30 or t[\'early_stopping_patience\'] != 6:\n        raise ValueError(\'Team protocol: 1..30 epochs and patience=6.\')\n    expected_world = int(cfg.get(\'distributed\', {}).get(\'world_size\', 1))\n    if ctx.world != expected_world:\n        raise ValueError(f\'Config expects {expected_world} training process(es) but {ctx.world} were started.\')\n    batch, effective = (cfg[\'data\'][\'batch_size\'], t.get(\'effective_batch_size\', 32))\n    per_step = batch * ctx.world\n    if batch < 1 or effective < per_step or effective % per_step:\n        raise ValueError(\'Effective batch size must be a positive multiple of per-GPU microbatch x GPU count.\')\n    accumulation = effective // per_step\n    out = Path(cfg[\'output\'][\'results_dir\']) / cfg[\'run_name\']\n    if out.exists() and (not resume):\n        raise FileExistsError(f\'Use a new run_name or --resume: {out}\')\n    # Rank 0 re-hashes train/val images (the audit\'s tamper check); the others only parse the manifest.\n    rows, audit = load_audited_manifest(cfg[\'data\'][\'manifest\'], cfg[\'data\'][\'root\'], cfg[\'data\'].get(\'near_duplicate_review\'), (\'train\', \'val\') if ctx.main else ())\n    dist_utils.barrier(ctx)\n    device = ctx.device\n    set_seed(cfg[\'seed\'])\n    if ctx.main:\n        model = build_model(cfg, pretrained=False if resume else None)  # rank 0 downloads weights once\n    dist_utils.barrier(ctx)\n    if not ctx.main:\n        model = build_model(cfg, pretrained=False if resume else None)  # others read the local cache\n    if ctx.world > 1 and device.type == \'cuda\' and any((isinstance(m, torch.nn.modules.batchnorm._BatchNorm) for m in model.modules())):\n        model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)\n    model = model.to(device)\n    if ctx.world > 1:\n        torch.manual_seed(cfg[\'seed\'] + 1000 * ctx.rank)  # independent dropout/augmentation noise per GPU\n    preproc = preprocessing_for(model, cfg)\n    cache = open_cache(preproc, audit[\'manifest_sha256\'])\n    if ctx.main:\n        print(f"Processes: {ctx.world} | per-GPU microbatch: {batch} | accumulation: {accumulation} | effective batch: {effective} | presize cache: {\'on\' if cache else \'off\'}", flush=True)\n    generator = torch.Generator().manual_seed(cfg[\'seed\'] + ctx.rank)\n    workers = cfg[\'data\'].get(\'num_workers\', 0)\n    limit = 32 if cfg.get(\'smoke\') else None\n\n    def loader_for(part):\n        dataset = ManifestDataset(rows, cfg[\'data\'][\'root\'], part, preproc, train=part == \'train\', limit=limit, cache=cache)\n        if part == \'train\':\n            sampler = DistributedSampler(dataset, num_replicas=ctx.world, rank=ctx.rank, shuffle=True, seed=cfg[\'seed\'], drop_last=False) if ctx.world > 1 else None\n        else:\n            sampler = ShardSampler(len(dataset), ctx.rank, ctx.world)\n        extra = {\'prefetch_factor\': 4} if workers else {}\n        return DataLoader(dataset, batch_size=batch, sampler=sampler, shuffle=part == \'train\' and sampler is None, generator=generator, num_workers=workers, pin_memory=device.type == \'cuda\', drop_last=False, **extra)\n    loaders = {part: loader_for(part) for part in (\'train\', \'val\')}\n    amp = bool(t.get(\'mixed_precision\', True) and device.type == \'cuda\')\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=amp)\n    history, best_loss, stale, elapsed, start_epoch = ([], float(\'inf\'), 0, 0.0, 1)\n    checkpoint = None\n    if resume:\n        checkpoint = torch.load(resume, map_location=\'cpu\', weights_only=True)\n        if checkpoint[\'config\'] != cfg or checkpoint[\'metadata\'][\'manifest_sha256\'] != audit[\'manifest_sha256\']:\n            raise ValueError(\'Resume config/manifest mismatch.\')\n        model.load_state_dict(checkpoint[\'model_state\'])\n        history, best_loss, stale = (checkpoint[\'history\'], checkpoint[\'best_loss\'], checkpoint[\'stale\'])\n        elapsed, start_epoch = (checkpoint[\'elapsed\'], checkpoint[\'epoch\'] + 1)\n        scaler.load_state_dict(checkpoint[\'scaler\'])\n        restore_rng(checkpoint[\'rng\'], generator, ctx)\n        if not (out / \'checkpoints/best_model.pt\').exists():\n            raise FileNotFoundError(\'Resume needs the original best checkpoint as well as last.pt.\')\n        if checkpoint[\'phase\'] == \'finetune\' and stale >= t[\'early_stopping_patience\']:\n            if ctx.main:\n                print(\'This run already reached early stopping; no further updates are allowed.\')\n            dist_utils.barrier(ctx)\n            return out\n    ckpt_dir = out / \'checkpoints\'\n    gpu_names = dist_utils.all_gather_object(ctx, torch.cuda.get_device_name(device) if device.type == \'cuda\' else platform.machine())\n    if ctx.main:\n        out.mkdir(parents=True, exist_ok=True)\n        ckpt_dir.mkdir(exist_ok=True)\n        (out / \'run_config_used.json\').write_text(json.dumps(cfg, indent=2))\n        versions = {name: importlib.metadata.version(name) for name in (\'torch\', \'torchvision\', \'timm\', \'numpy\', \'pandas\', \'scikit-learn\', \'Pillow\', \'PyYAML\')}\n        environment = {\'python\': platform.python_version(), \'packages\': versions, \'device\': str(device), \'cuda\': torch.version.cuda, \'gpu\': gpu_names[0], \'gpus\': gpu_names, \'world_size\': ctx.world, \'per_device_batch\': batch, \'gradient_accumulation\': accumulation, \'presize_cache\': cache is not None}\n        try:\n            environment[\'git_commit\'] = subprocess.check_output([\'git\', \'rev-parse\', \'HEAD\'], text=True, stderr=subprocess.DEVNULL).strip()\n            environment[\'git_dirty\'] = bool(subprocess.check_output([\'git\', \'status\', \'--porcelain\'], text=True, stderr=subprocess.DEVNULL).strip())\n        except (OSError, subprocess.CalledProcessError):\n            environment[\'git_commit\'] = None\n        (out / \'environment.json\').write_text(json.dumps(environment, indent=2))\n    metadata = {\'model_name\': cfg[\'model_name\'], \'run_name\': cfg[\'run_name\'], \'manifest_sha256\': audit[\'manifest_sha256\'], \'dataset_version\': audit[\'dataset_version\'], \'preprocessing\': preproc, \'label_mapping\': {\'real\': 0, \'fake\': 1}, \'threshold\': 0.5, \'selection\': \'validation_loss\', \'smoke\': cfg.get(\'smoke\', False), \'model_config\': cfg[\'model\'], \'module\': cfg[\'module\'], \'seed\': cfg[\'seed\'], \'world_size\': ctx.world}\n    current_phase, optimizer, scheduler, network = (None, None, None, None)\n    epoch_wall = []\n    reason = dist_utils.broadcast_object(ctx, preflight_reason(history, start_epoch, t[\'epochs\'], time.perf_counter() - invocation_start) if ctx.main else None)\n    if reason:\n        if ctx.main:\n            save_pause(out, ckpt_dir, start_epoch - 1, reason, time.perf_counter() - invocation_start)\n        dist_utils.barrier(ctx)\n        return out\n    for epoch in range(start_epoch, t[\'epochs\'] + 1):\n        warmup = epoch <= t.get(\'warmup_epochs\', 0)\n        phase = \'warmup\' if warmup else \'finetune\'\n        if phase != current_phase:\n            phase_parameters(model, cfg, warmup)\n            network = None\n            network = wrap(model, ctx)\n            lr = t.get(\'warmup_lr\', t[\'lr\']) if warmup else t[\'lr\']\n            cls = torch.optim.Adam if t.get(\'optimizer\') == \'adam\' else torch.optim.AdamW\n            optimizer = cls(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=t[\'weight_decay\'])\n            total_steps = max(1, t[\'epochs\'] * math.ceil(len(loaders[\'train\']) / accumulation))\n            warm_steps = int(total_steps * t.get(\'warmup_steps_pct\', 0))\n\n            def schedule(step):\n                if step < warm_steps:\n                    return (step + 1) / max(1, warm_steps)\n                return max(0.0, (total_steps - step) / max(1, total_steps - warm_steps))\n            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\n            if checkpoint and checkpoint[\'phase\'] == phase:\n                optimizer.load_state_dict(checkpoint[\'optimizer\'])\n                scheduler.load_state_dict(checkpoint[\'scheduler\'])\n            elif current_phase is not None or checkpoint:\n                stale = 0\n            current_phase = phase\n        if isinstance(loaders[\'train\'].sampler, DistributedSampler):\n            loaders[\'train\'].sampler.set_epoch(epoch)\n        tick = time.perf_counter()\n        train_loss, train_metrics, train_stats = run_epoch(network, loaders[\'train\'], device, ctx, optimizer, scheduler, scaler, accumulation, t.get(\'grad_clip_norm\', 1.0), amp, log_every=t.get(\'progress_every\', 0))\n        val_loss, val_metrics, val_stats = run_epoch(network, loaders[\'val\'], device, ctx, amp=amp, log_every=t.get(\'progress_every\', 0))\n        elapsed += time.perf_counter() - tick\n        improved = val_loss < best_loss\n        if improved:\n            best_loss, stale = (val_loss, 0)\n        else:\n            stale += 1\n        rng_all = dist_utils.all_gather_object(ctx, rng_state(generator, ctx))\n        stop = dist_utils.broadcast_object(ctx, not warmup and stale >= t[\'early_stopping_patience\'])\n        if ctx.main:\n            history.append({\'epoch\': epoch, \'phase\': phase, \'train_loss\': train_loss, \'val_loss\': val_loss, \'train_accuracy\': train_metrics[\'accuracy\'], \'val_accuracy\': val_metrics[\'accuracy\'], \'train_f1\': train_metrics[\'f1_score\'], \'val_f1\': val_metrics[\'f1_score\'], \'trainable_parameters\': sum((p.numel() for p in model.parameters() if p.requires_grad)), \'lr\': optimizer.param_groups[0][\'lr\'], \'elapsed_seconds\': elapsed, \'train_precision\': train_metrics[\'precision\'], \'train_recall\': train_metrics[\'recall\'], \'train_roc_auc\': train_metrics[\'roc_auc\'], \'train_average_precision\': train_metrics[\'average_precision\'], \'train_balanced_accuracy\': train_metrics[\'balanced_accuracy\'], \'train_specificity\': train_metrics[\'specificity\'], \'train_mcc\': train_metrics[\'mcc\'], \'val_precision\': val_metrics[\'precision\'], \'val_recall\': val_metrics[\'recall\'], \'val_roc_auc\': val_metrics[\'roc_auc\'], \'val_average_precision\': val_metrics[\'average_precision\'], \'val_balanced_accuracy\': val_metrics[\'balanced_accuracy\'], \'val_specificity\': val_metrics[\'specificity\'], \'val_mcc\': val_metrics[\'mcc\'], \'train_images_per_second\': train_stats[\'images_per_second\'], \'train_data_wait_fraction\': train_stats[\'data_wait_fraction\'], \'world_size\': ctx.world})\n            metadata.update(training_time_seconds=elapsed, total_parameters=sum((p.numel() for p in model.parameters())), trainable_parameters=sum((p.numel() for p in model.parameters() if p.requires_grad)))\n            (out / \'run_metadata.json\').write_text(json.dumps(metadata, indent=2))\n            if improved:\n                torch.save({\'model_state\': model.state_dict(), \'metadata\': dict(metadata, best_epoch=epoch)}, ckpt_dir / \'best_model.tmp\')\n                (ckpt_dir / \'best_model.tmp\').replace(ckpt_dir / \'best_model.pt\')\n            state = {\'model_state\': model.state_dict(), \'metadata\': metadata, \'config\': cfg, \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(), \'scaler\': scaler.state_dict(), \'epoch\': epoch, \'phase\': phase, \'history\': history, \'best_loss\': best_loss, \'stale\': stale, \'elapsed\': elapsed, \'rng\': rng_all if ctx.world > 1 else rng_all[0]}\n            temporary = ckpt_dir / \'last.tmp\'\n            torch.save(state, temporary)\n            temporary.replace(ckpt_dir / \'last.pt\')\n            pd.DataFrame(history).to_csv(out / \'training_history.csv\', index=False)\n            print(f"epoch={epoch} phase={phase} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_metrics[\'accuracy\']:.4f} train_img/s={train_stats[\'images_per_second\']:.1f} data_wait={100 * train_stats[\'data_wait_fraction\']:.0f}% epoch_min={(time.perf_counter() - tick) / 60:.1f}", flush=True)\n        if stop:\n            break\n        if ctx.main:\n            learning_curves(history, out / \'learning_curves.png\')\n        epoch_wall.append(time.perf_counter() - tick)\n        reason = dist_utils.broadcast_object(ctx, pause_reason(epoch, start_epoch, t[\'epochs\'], epoch_wall, time.perf_counter() - invocation_start) if ctx.main else None)\n        if reason:\n            if ctx.main:\n                save_pause(out, ckpt_dir, epoch, reason, time.perf_counter() - invocation_start)\n            dist_utils.barrier(ctx)\n            return out\n    if ctx.main:\n        if not history:\n            raise ValueError(\'No training history; check resume epoch.\')\n        learning_curves(history, out / \'learning_curves.png\')\n        best = torch.load(ckpt_dir / \'best_model.pt\', map_location=\'cpu\', weights_only=True)\n        best[\'metadata\'][\'training_time_seconds\'] = elapsed\n        torch.save(best, ckpt_dir / \'best_model.tmp\')\n        (ckpt_dir / \'best_model.tmp\').replace(ckpt_dir / \'best_model.pt\')\n        (out / \'training_complete.json\').write_text(json.dumps({\'status\': \'smoke_only\' if cfg.get(\'smoke\') else \'trained\', \'epochs\': len(history), \'seconds\': elapsed}, indent=2))\n        (out / \'training_paused.json\').unlink(missing_ok=True)\n    dist_utils.barrier(ctx)\n    return out\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'-c\', \'--config\', default=\'configs/vit.yaml\')\n    parser.add_argument(\'--data-root\')\n    parser.add_argument(\'--manifest\')\n    parser.add_argument(\'--device\')\n    parser.add_argument(\'--resume\')\n    parser.add_argument(\'--smoke\', action=\'store_true\')\n    args = parser.parse_args()\n    cfg = load_config(args.config)\n    for key, value in ((\'root\', args.data_root), (\'manifest\', args.manifest)):\n        if value:\n            cfg[\'data\'][key] = value\n    if args.device:\n        cfg[\'device\'] = args.device\n    if args.smoke:\n        cfg.update(smoke=True, run_name=cfg[\'run_name\'] + \'_smoke\')\n        cfg[\'train\'].update(epochs=1, warmup_epochs=0)\n        cfg[\'data\'].update(batch_size=2, num_workers=0)\n    print(f\'Training artifacts: {train(cfg, args.resume)}. Test sets have not been evaluated.\')\n\n\ndef chunk_budget():\n    """Per-invocation limits that let a Kaggle saved run finish and publish its outputs."""\n    epochs = int(os.environ.get("WISH_EPOCHS_PER_INVOCATION", "3"))\n    if epochs < 1:\n        raise ValueError("WISH_EPOCHS_PER_INVOCATION must be positive")\n    seconds = float(os.environ.get("WISH_TIME_BUDGET_SECONDS", "0"))\n    if seconds < 0:\n        raise ValueError("WISH_TIME_BUDGET_SECONDS must not be negative")\n    return (epochs, seconds)\n\ndef epoch_estimate(spans):\n    """Conservative next-epoch cost from measured epochs only; never an invented constant."""\n    recent = [float(span) for span in spans[-3:] if span is not None and float(span) > 0]\n    if not recent:\n        return None\n    return max(recent) * 1.15 + 120.0\n\ndef history_epoch_spans(history):\n    times = [float(row["elapsed_seconds"]) for row in history if row.get("elapsed_seconds") is not None]\n    return [b - a for a, b in zip(times, times[1:])] if len(times) > 1 else times\n\ndef pause_reason(epoch, start_epoch, total_epochs, spans, spent):\n    epochs, seconds = chunk_budget()\n    if epoch >= total_epochs:\n        return None\n    if epoch - start_epoch + 1 >= epochs:\n        return "saved_run_epoch_budget"\n    estimate = epoch_estimate(spans)\n    if seconds and estimate is not None and spent + estimate > seconds:\n        return "saved_run_time_budget"\n    return None\n\ndef preflight_reason(history, start_epoch, total_epochs, spent):\n    epochs, seconds = chunk_budget()\n    if start_epoch > total_epochs or not seconds:\n        return None\n    estimate = epoch_estimate(history_epoch_spans(history))\n    if estimate is not None and spent + estimate > seconds:\n        return "insufficient_time_budget"\n    return None\n\ndef save_pause(out, ckpt_dir, epoch, reason, spent):\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "training_paused.json").write_text(json.dumps({"status": "paused_not_complete", "last_epoch": epoch,\n        "reason": reason, "seconds_used_this_invocation": round(spent, 1), "resume": str(ckpt_dir / "last.pt")}, indent=2))\n    print(f"Training chunk saved ({reason}). Resume the same experiment from these saved outputs.", flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n',
    'models/vit/evaluate_crossgen.py': '"""Freeze selected checkpoints, then run a strict four-model final evaluation."""\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom models.vit.manifest import load_audited_manifest, sha256_file\nfrom models.vit.reporting import compute_metrics, make_figures\nfrom models.vit.runtime import (ManifestDataset, device_for, load_checkpoint, load_config,\n                                probabilities)\n\nREQUIRED = {"custom_cnn", "resnet50", "efficientnetv2", "vit_b16"}\n\n\ndef validate_metadata(metadata, cfg, checksum):\n    if metadata.get("manifest_sha256") != checksum:\n        raise ValueError("Checkpoint was not trained on the frozen shared manifest.")\n    if metadata.get("smoke", True) or metadata.get("selection") != "validation_loss":\n        raise ValueError("Final evaluation requires a non-smoke, validation-selected checkpoint.")\n    if metadata.get("label_mapping") != {"real": 0, "fake": 1} or metadata.get("threshold") != 0.5:\n        raise ValueError("Checkpoint label mapping / threshold conflicts with team protocol.")\n    if metadata.get("model_config") != cfg["model"] or metadata.get("module") != cfg["module"]:\n        raise ValueError("Checkpoint architecture/config mismatch.")\n    if metadata.get("preprocessing", {}).get("image_size") != 224:\n        raise ValueError("Final comparison requires 224x224 preprocessing.")\n    for key in ("training_time_seconds", "total_parameters", "trainable_parameters", "run_name"):\n        if key not in metadata:\n            raise ValueError(f"Checkpoint metadata missing {key}; obtain it from the model owner.")\n\n\ndef freeze(config, destination):\n    cfg = load_config(config)\n    if not REQUIRED.issubset(cfg["models"]):\n        raise ValueError("Final comparison requires all four core models.")\n    destination = Path(destination)\n    if destination.exists():\n        raise FileExistsError("Do not overwrite a frozen experiment.")\n    _, audit = load_audited_manifest(cfg["manifest"], cfg["data_root"],\n                                     cfg.get("near_duplicate_review"), verify_splits=())\n    entries = {}\n    for name, entry in cfg["models"].items():\n        if not Path(entry["checkpoint"]).is_file():\n            raise FileNotFoundError(f"{name}: missing checkpoint {entry[\'checkpoint\']}")\n        model_cfg = load_config(entry["config"])\n        bundle = torch.load(entry["checkpoint"], map_location="cpu", weights_only=True)\n        metadata = bundle.get("metadata")\n        if metadata is None:\n            metadata = json.loads(Path(entry["checkpoint"] + ".json").read_text())\n        validate_metadata(metadata, model_cfg, audit["manifest_sha256"])\n        if model_cfg["model_name"] != name:\n            raise ValueError(f"Model name mismatch for {name}")\n        entries[name] = {"config": model_cfg, "checkpoint": entry["checkpoint"],\n                         "checkpoint_sha256": sha256_file(entry["checkpoint"]), "metadata": metadata}\n        del bundle\n    protocol = {"frozen_at_utc": datetime.now(timezone.utc).isoformat(),\n                "manifest": cfg["manifest"], "manifest_sha256": audit["manifest_sha256"],\n                "data_root": cfg["data_root"], "near_duplicate_review": cfg.get("near_duplicate_review"),\n                "threshold": 0.5, "models": entries}\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(json.dumps(protocol, indent=2) + "\\n")\n    return protocol\n\n\ndef synchronize(device):\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    elif device.type == "mps":\n        torch.mps.synchronize()\n\n\n@torch.inference_mode()\ndef predict_loader(model, loader, device, name, run_name):\n    model.eval()\n    predictions, seconds = [], 0.0\n    for images, labels, indices in loader:\n        images = images.to(device)\n        synchronize(device)\n        start = time.perf_counter()\n        scores = probabilities(model(images))\n        synchronize(device)\n        seconds += time.perf_counter() - start\n        for label, score, index in zip(labels.tolist(), scores.cpu().tolist(), indices.tolist()):\n            row = loader.dataset.rows[index]\n            predictions.append({"model": name, "run_name": run_name, "image_id": row["sha256"],\n                                "filepath": row["filepath"], "label": label, "probability": score,\n                                "predicted_label": int(score >= 0.5), "split": row["split"],\n                                "generator": row["source"]})\n    return predictions, seconds * 1000 / len(predictions)\n\n\n@torch.inference_mode()\ndef single_image_latency(model, dataset, device, repeats=30):\n    image = dataset[0][0].unsqueeze(0).to(device)\n    for _ in range(5):\n        model(image)\n    synchronize(device)\n    tick = time.perf_counter()\n    for _ in range(repeats):\n        model(image)\n    synchronize(device)\n    return 1000 * (time.perf_counter() - tick) / repeats\n\n\ndef evaluate(frozen, output, device="auto", batch_size=32):\n    protocol = json.loads(Path(frozen).read_text())\n    if not REQUIRED.issubset(protocol["models"]) or protocol.get("threshold") != 0.5:\n        raise ValueError("Invalid frozen four-model protocol.")\n    rows, audit = load_audited_manifest(protocol["manifest"], protocol["data_root"],\n                                       protocol.get("near_duplicate_review"))\n    if protocol["manifest_sha256"] != audit["manifest_sha256"]:\n        raise ValueError("Manifest changed after experiment freeze.")\n    for name, entry in protocol["models"].items():\n        if sha256_file(entry["checkpoint"]) != entry["checkpoint_sha256"]:\n            raise ValueError(f"Checkpoint changed after freeze: {name}")\n        validate_metadata(entry["metadata"], entry["config"], audit["manifest_sha256"])\n    output = Path(output)\n    if output.exists():\n        raise FileExistsError("Use a new evaluation directory; never overwrite final evidence.")\n    output.mkdir(parents=True)\n    status = output / "status.json"\n    status.write_text(json.dumps({"status": "running", "frozen_sha256": sha256_file(frozen)}))\n    device = device_for(device)\n    all_predictions, comparison, expected_ids = [], [], {}\n    try:\n        for name, entry in protocol["models"].items():\n            print(f"Evaluating frozen {name}", flush=True)\n            model, metadata = load_checkpoint(entry["config"], entry["checkpoint"], device)\n            if metadata != entry["metadata"]:\n                raise ValueError("Checkpoint metadata changed since freeze.")\n            metrics, times = {}, {}\n            model_predictions = []\n            for split in ("test", "cross_gen"):\n                dataset = ManifestDataset(rows, protocol["data_root"], split, metadata["preprocessing"])\n                loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)\n                predictions, batch_latency = predict_loader(model, loader, device, name, metadata["run_name"])\n                ids = [p["image_id"] for p in predictions]\n                if split in expected_ids and expected_ids[split] != ids:\n                    raise ValueError("Models did not evaluate identical ordered sample lists.")\n                expected_ids[split] = ids\n                metrics[split] = compute_metrics([p["label"] for p in predictions],\n                                                 [p["probability"] for p in predictions])\n                times[split] = {"batch_ms_per_image": batch_latency,\n                                "single_image_ms": single_image_latency(model, dataset, device)}\n                all_predictions.extend(predictions)\n                model_predictions.extend(predictions)\n            model_dir = output / name\n            model_dir.mkdir()\n            pd.DataFrame(model_predictions).to_csv(model_dir / "predictions.csv", index=False)\n            result = {"model_name": name, "test_metrics": metrics["test"],\n                      "cross_gen_metrics": metrics["cross_gen"], "timing": times,\n                      "total_train_time_seconds": metadata["training_time_seconds"],\n                      "total_parameters": metadata["total_parameters"],\n                      "trainable_parameters": metadata["trainable_parameters"],\n                      "generalization_gap_accuracy": metrics["test"]["accuracy"] - metrics["cross_gen"]["accuracy"],\n                      "generalization_gap_f1": metrics["test"]["f1_score"] - metrics["cross_gen"]["f1_score"]}\n            (model_dir / "results.json").write_text(json.dumps(result, indent=2, allow_nan=False))\n            comparison.append({"model": name, "primary_accuracy": metrics["test"]["accuracy"],\n                               "cross_accuracy": metrics["cross_gen"]["accuracy"],\n                               "primary_f1": metrics["test"]["f1_score"],\n                               "cross_f1": metrics["cross_gen"]["f1_score"],\n                               "cross_auc": metrics["cross_gen"]["roc_auc"],\n                               "accuracy_drop_pp": 100 * result["generalization_gap_accuracy"],\n                               "f1_drop": result["generalization_gap_f1"],\n                               "parameters": metadata["total_parameters"],\n                               "training_seconds": metadata["training_time_seconds"],\n                               "latency_ms": times["test"]["single_image_ms"]})\n            del model\n            gc.collect()\n        predictions_path = output / "predictions.csv"\n        pd.DataFrame(all_predictions).to_csv(predictions_path, index=False)\n        pd.DataFrame(comparison).to_csv(output / "model_comparison.csv", index=False)\n        make_figures(predictions_path, output / "figures")\n        status.write_text(json.dumps({"status": "complete", "models": list(protocol["models"]),\n                                     "frozen_sha256": sha256_file(frozen), "device": str(device)}, indent=2))\n    except Exception as error:\n        status.write_text(json.dumps({"status": "failed_not_final", "error": str(error)}, indent=2))\n        raise\n    return output\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest="command", required=True)\n    freeze_parser = sub.add_parser("freeze")\n    freeze_parser.add_argument("-c", "--config", default="configs/crossgen_harness.yaml")\n    freeze_parser.add_argument("--output", default="results/comparison/frozen.json")\n    run_parser = sub.add_parser("run")\n    run_parser.add_argument("--frozen", default="results/comparison/frozen.json")\n    run_parser.add_argument("--output", default="results/comparison/final")\n    run_parser.add_argument("--device", default="auto")\n    run_parser.add_argument("--batch-size", type=int, default=32)\n    plots_parser = sub.add_parser("plots")\n    plots_parser.add_argument("--predictions", required=True)\n    plots_parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n    if args.command == "freeze":\n        freeze(args.config, args.output)\n        print(f"Protocol frozen: {args.output}. No test predictions were computed.")\n    elif args.command == "run":\n        print(evaluate(args.frozen, args.output, args.device, args.batch_size))\n    else:\n        make_figures(args.predictions, args.output)\n\n\nif __name__ == "__main__":\n    main()\n',
    'models/vit/wish_standalone.py': '"""Independent Wish experiment: content-grouped splits and sealed ViT evaluation.\n\nThis does not replace the team\'s existing split contract. Other models must use\nthe exported manifest before their results can form a controlled comparison.\n"""\nfrom __future__ import annotations\nimport io\nfrom models.vit.audit_progress import discover_images, inspect_inventory, verify_inventory, timed_split_assignment\nimport csv\nimport hashlib\nimport json\nimport os\nimport random\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom PIL import Image, ImageOps\nfrom models.vit.manifest import DATASET, FIELDS, load_audited_manifest, sha256_file, source_from_name, validate_rows\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.webp\'}\n\ndef find_root(directory):\n    roots = []\n    for current, directories, _ in os.walk(directory):\n        if \'Real\' in directories and \'Fake\' in directories:\n            roots.append(Path(current).resolve())\n            directories[:] = []\n    if len(roots) != 1:\n        raise ValueError(f\'Expected one Real/Fake image root under {directory}; found {roots}\')\n    return roots[0]\n\ndef write_csv(path, rows, fields):\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction=\'ignore\')\n        writer.writeheader()\n        writer.writerows(rows)\n\ndef inspect_image(item):\n    root, relative = item\n    label, source = source_from_name(relative)\n    if Path(relative).parts[0] != (\'Real\' if label == 0 else \'Fake\'):\n        raise ValueError(f\'Folder and filename labels disagree: {relative}\')\n    path = root / relative\n    try:\n        content = path.read_bytes()\n        with Image.open(io.BytesIO(content)) as image:\n            image.load()\n            rgb = ImageOps.exif_transpose(image).convert(\'RGB\')\n            pixels = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()\n            small = list(rgb.convert(\'L\').resize((9, 8)).tobytes())\n            dhash = sum((int(small[y * 9 + x] > small[y * 9 + x + 1]) << y * 8 + x for y in range(8) for x in range(8)))\n    except Exception as error:\n        raise ValueError(f\'Unreadable image {relative}: {error}\') from error\n    return {\'filepath\': relative, \'label\': label, \'source\': source, \'identity\': \'\', \'sha256\': hashlib.sha256(content).hexdigest(), \'pixel_sha256\': pixels, \'dhash\': f\'{dhash:016x}\'}\n\ndef group_content(rows, distance=4):\n    """Union all dHash neighbors; multi-index lookup avoids all-pairs comparison."""\n    if not 0 <= distance <= 8:\n        raise ValueError(\'Near-duplicate distance must be in 0..8\')\n    parent = list(range(len(rows)))\n\n    def find(index):\n        while parent[index] != index:\n            parent[index] = parent[parent[index]]\n            index = parent[index]\n        return index\n\n    def union(a, b):\n        a, b = (find(a), find(b))\n        if a != b:\n            parent[max(a, b)] = min(a, b)\n    seen, buckets, pairs = ({}, defaultdict(list), [])\n    for index, row in enumerate(rows):\n        for kind in (\'sha256\', \'pixel_sha256\'):\n            key = (kind, row[kind])\n            if key in seen:\n                previous = seen[key]\n                if rows[previous][\'label\'] != row[\'label\']:\n                    raise ValueError(f"Exact duplicate has conflicting labels: {row[\'filepath\']}")\n                union(index, previous)\n            seen[key] = index\n        value = int(row[\'dhash\'], 16)\n        keys, candidates = ([], set())\n        for part in range(distance + 1):\n            start, end = (64 * part // (distance + 1), 64 * (part + 1) // (distance + 1))\n            key = (part, value >> start & (1 << end - start) - 1)\n            keys.append(key)\n            candidates.update(buckets[key])\n        for previous in sorted(candidates):\n            difference = (value ^ int(rows[previous][\'dhash\'], 16)).bit_count()\n            if difference <= distance:\n                union(index, previous)\n                pairs.append({\'filepath_a\': rows[previous][\'filepath\'], \'filepath_b\': row[\'filepath\'], \'distance\': difference})\n        for key in keys:\n            buckets[key].append(index)\n        if (index + 1) % 10000 == 0:\n            print(f\'Grouped {index + 1:,}/{len(rows):,} images\', flush=True)\n    groups = defaultdict(list)\n    for index, row in enumerate(rows):\n        groups[find(index)].append(dict(row))\n    return (list(groups.values()), pairs)\n\ndef take_groups(groups, target, leave=0):\n    selected, count = ([], 0)\n    limit = len(groups) - leave\n    while len(selected) < limit and count < target:\n        group = groups[len(selected)]\n        selected.append(group)\n        count += len(group)\n    del groups[:len(selected)]\n    return selected\n\ndef assign_splits(rows, seed=42, distance=4):\n    """Deduplicate before splitting; keep perceptual components in one domain."""\n    rows = sorted(rows, key=lambda row: row[\'filepath\'])\n    excluded = [dict(row, reason=\'unspecified_ai_source\') for row in rows if row[\'source\'] == \'AiGenImage\']\n    groups, pairs = group_content([r for r in rows if r[\'source\'] != \'AiGenImage\'], distance)\n    strata = defaultdict(list)\n    assigned = []\n    for group in groups:\n        sources, labels = ({r[\'source\'] for r in group}, {r[\'label\'] for r in group})\n        if len(labels) > 1 or {\'StyleGAN\', \'StableDiffusion\'}.issubset(sources):\n            excluded.extend((dict(row, reason=\'ambiguous_cross_source_perceptual_group\') for row in group))\n            continue\n        unique, seen = ([], set())\n        for row in group:\n            key = row[\'pixel_sha256\']\n            if key in seen:\n                excluded.append(dict(row, reason=\'exact_duplicate_removed_before_split\'))\n            else:\n                seen.add(key)\n                unique.append(row)\n        group_id = hashlib.sha256(\'\\n\'.join((r[\'filepath\'] for r in group)).encode()).hexdigest()\n        for row in unique:\n            row[\'group_id\'] = group_id\n        if sources == {\'StableDiffusion\'}:\n            assigned.extend((dict(row, split=\'cross_gen\') for row in unique))\n        else:\n            stratum = Counter((r[\'source\'] for r in unique)).most_common(1)[0][0]\n            strata[stratum].append(unique)\n    if not assigned or not all((strata[key] for key in (\'FFHQ\', \'CelebA\', \'StyleGAN\'))):\n        raise ValueError(\'Need FFHQ, CelebA, StyleGAN and StableDiffusion after duplicate filtering.\')\n    rng = random.Random(seed)\n    real_count = sum((len(g) for key in (\'FFHQ\', \'CelebA\') for g in strata[key]))\n    cross_fraction = len(assigned) / real_count\n    if cross_fraction >= 0.5:\n        raise ValueError(\'Too few real controls for this cross-generator protocol.\')\n    for source in (\'FFHQ\', \'CelebA\', \'StyleGAN\'):\n        groups = strata[source]\n        rng.shuffle(groups)\n        if source != \'StyleGAN\':\n            controls = take_groups(groups, round(sum(map(len, groups)) * cross_fraction), leave=3)\n            assigned.extend((dict(row, split=\'cross_gen\') for group in controls for row in group))\n        if len(groups) < 3:\n            raise ValueError(f\'Too few independent content groups for {source}\')\n        total = sum(map(len, groups))\n        train = take_groups(groups, round(total * 0.7), leave=2)\n        val = take_groups(groups, round(total * 0.15), leave=1)\n        for split, items in ((\'train\', train), (\'val\', val), (\'test\', groups)):\n            assigned.extend((dict(row, split=split) for group in items for row in group))\n    assigned.sort(key=lambda row: row[\'filepath\'])\n    validate_rows(assigned)\n    by_group = defaultdict(set)\n    for row in assigned:\n        by_group[row[\'group_id\']].add(row[\'split\'])\n    if any((len(parts) != 1 for parts in by_group.values())):\n        raise AssertionError(\'Content group leaked between splits\')\n    return (assigned, excluded, pairs)\n\ndef prepare(root, output, seed=42, provenance=None, workers=4):\n    root, output = (Path(root), Path(output))\n    files = discover_images(root, EXTENSIONS)\n    if not files:\n        raise ValueError(f\'No images under {root}\')\n    provenance = provenance or {\'version\': \'unreported_attached_snapshot\'}\n    settings = {\'seed\': seed, \'near_distance\': 4, \'provenance\': provenance, \'protocol\': \'wish_independent_v1\'}\n    if output.exists():\n        if not (output / \'audit.json\').is_file():\n            raise ValueError(\'Incomplete audit directory. Use a new OUTPUT_ROOT.\')\n        audit = json.loads((output / \'audit.json\').read_text())\n        if audit.get(\'independent_settings\') != settings:\n            raise ValueError(\'Data preparation settings changed. Use a new OUTPUT_ROOT.\')\n        with (output / \'inventory.csv\').open() as handle:\n            inventory = list(csv.DictReader(handle))\n        if sha256_file(output / \'inventory.csv\') != audit[\'inventory_sha256\']:\n            raise ValueError(\'Inventory checksum mismatch\')\n        if [r[\'filepath\'] for r in inventory] != files:\n            raise ValueError(\'Dataset file list changed. Use a new OUTPUT_ROOT.\')\n        verify_inventory(root, inventory, sha256_file, workers)\n        load_audited_manifest(output / \'manifest.csv\', root, verify_splits=())\n        print(\'Reusing verified, unchanged split manifest\', flush=True)\n        return audit\n    print(f\'Reading and hashing {len(files):,} images; no model evaluation occurs here.\', flush=True)\n    inventory = inspect_inventory(root, files, inspect_image, workers)\n    rows, excluded, pairs = timed_split_assignment(assign_splits, inventory, seed)\n    output.mkdir(parents=True)\n    write_csv(output / \'inventory.csv\', inventory, FIELDS)\n    write_csv(output / \'manifest.csv\', rows, (*FIELDS, \'group_id\'))\n    write_csv(output / \'excluded.csv\', excluded, (*FIELDS, \'reason\'))\n    write_csv(output / \'near_duplicate_candidates.csv\', pairs, (\'filepath_a\', \'filepath_b\', \'distance\'))\n    for split in (\'train\', \'val\', \'test\', \'cross_gen\'):\n        write_csv(output / f\'{split}.csv\', [r for r in rows if r[\'split\'] == split], (*FIELDS, \'group_id\'))\n    write_csv(output / \'near_duplicates.csv\', [], (\'filepath_a\', \'filepath_b\', \'distance\'))\n    snapshot = sha256_file(output / \'inventory.csv\')\n    audit = {\'dataset\': DATASET, \'dataset_version\': f\'content-sha256:{snapshot}\', \'manifest_sha256\': sha256_file(output / \'manifest.csv\'), \'inventory_sha256\': snapshot, \'independent_settings\': settings, \'near_duplicate_pairs\': 0, \'detected_near_pairs_before_grouping\': len(pairs), \'near_distance\': 4, \'identity_metadata_available\': False, \'status\': \'passed\', \'counts\': dict(Counter((f"{r[\'split\']}:{r[\'source\']}:{r[\'label\']}" for r in rows))), \'exclusions\': dict(Counter((r[\'reason\'] for r in excluded))), \'limitations\': \'dHash is heuristic, not face identity verification; source/compression confounding remains.\'}\n    (output / \'audit.json\').write_text(json.dumps(audit, indent=2))\n    return audit\n\ndef train_or_resume(cfg):\n    """Entry for every torchrun process; one process per GPU joins the same run."""\n    from models.vit import distributed as dist_utils\n    from models.vit.train import train\n    ctx = dist_utils.setup(cfg.get(\'device\', \'auto\'))\n    try:\n        output = Path(cfg[\'output\'][\'results_dir\']) / cfg[\'run_name\']\n        complete = (output / \'training_complete.json\').exists()\n        last = output / \'checkpoints/last.pt\'\n        resume = str(last) if last.is_file() else None\n        dist_utils.barrier(ctx)  # every rank has read the run state before anything is written\n        if complete:\n            if json.loads((output / \'run_config_used.json\').read_text()) != cfg:\n                raise ValueError(\'Completed run has different settings. Choose a new RUN_NAME.\')\n            if ctx.main:\n                load_audited_manifest(cfg[\'data\'][\'manifest\'], cfg[\'data\'][\'root\'], verify_splits=(\'train\', \'val\'))\n                if not (output / \'checkpoints/best_model.pt\').is_file():\n                    raise FileNotFoundError(\'Completed run is missing its best checkpoint.\')\n                print(f\'Reusing completed run: {output}\', flush=True)\n            dist_utils.barrier(ctx)\n            return output\n        return train(cfg, resume=resume, ctx=ctx)\n    finally:\n        dist_utils.cleanup(ctx)\n\ndef evaluate_vit(cfg, checkpoint, output, device=\'cuda\'):\n    """Freeze one ViT before either held-out set; never imply a four-model result."""\n    import pandas as pd\n    import torch\n    from torch.utils.data import DataLoader\n    from models.vit.evaluate_crossgen import predict_loader, single_image_latency, validate_metadata\n    from models.vit.reporting import compute_metrics, make_figures\n    from models.vit.runtime import ManifestDataset, device_for, load_checkpoint\n    output, checkpoint = (Path(output), Path(checkpoint))\n    if output.exists():\n        raise FileExistsError(\'Evaluation already started here. Inspect saved results; do not rerun tests for tuning.\')\n    rows, audit = load_audited_manifest(cfg[\'data\'][\'manifest\'], cfg[\'data\'][\'root\'])\n    device = device_for(device)\n    model, metadata = load_checkpoint(cfg, checkpoint, device)\n    validate_metadata(metadata, cfg, audit[\'manifest_sha256\'])\n    output.mkdir(parents=True)\n    frozen = {\'checkpoint_sha256\': sha256_file(checkpoint), \'manifest_sha256\': audit[\'manifest_sha256\'], \'config\': cfg, \'metadata\': metadata, \'threshold\': 0.5, \'scope\': \'independent_vit_only_not_four_model_comparison\'}\n    (output / \'frozen.json\').write_text(json.dumps(frozen, indent=2))\n    status = output / \'status.json\'\n    status.write_text(json.dumps({\'status\': \'running\'}))\n    predictions, metrics, timings = ([], {}, {})\n    try:\n        for split in (\'test\', \'cross_gen\'):\n            dataset = ManifestDataset(rows, cfg[\'data\'][\'root\'], split, metadata[\'preprocessing\'])\n            loader = DataLoader(dataset, batch_size=cfg[\'data\'][\'batch_size\'], shuffle=False, num_workers=min(4, os.cpu_count() or 1), pin_memory=device.type == \'cuda\')\n            scores, batch_ms = predict_loader(model, loader, device, \'vit_b16\', cfg[\'run_name\'])\n            predictions.extend(scores)\n            metrics[split] = compute_metrics([r[\'label\'] for r in scores], [r[\'probability\'] for r in scores])\n            timings[split] = {\'batch_ms_per_image\': batch_ms, \'single_image_ms\': single_image_latency(model, dataset, device)}\n            print(f\'Completed sealed {split}: {len(scores):,} images\', flush=True)\n        pd.DataFrame(predictions).to_csv(output / \'predictions.csv\', index=False)\n        make_figures(output / \'predictions.csv\', output / \'figures\')\n        import matplotlib.pyplot as plt\n        fig, axes = plt.subplots(1, 2, figsize=(9, 4))\n        for axis, split in zip(axes, (\'test\', \'cross_gen\')):\n            cm = metrics[split][\'confusion_matrix\']\n            axis.imshow(cm, cmap=\'Blues\')\n            maximum = max(map(max, cm))\n            for y in (0, 1):\n                for x in (0, 1):\n                    axis.text(x, y, str(cm[y][x]), ha=\'center\', va=\'center\', color=\'white\' if cm[y][x] > maximum / 2 else \'black\')\n            axis.set(title=split, xlabel=\'Predicted\', ylabel=\'True\', xticks=[0, 1], yticks=[0, 1], xticklabels=[\'Real\', \'Fake\'], yticklabels=[\'Real\', \'Fake\'])\n        fig.tight_layout()\n        fig.savefig(output / \'figures/confusion_matrices_side_by_side.png\', dpi=180)\n        plt.close(fig)\n        result = {\'scope\': frozen[\'scope\'], \'metrics\': metrics, \'timing\': timings, \'timing_scope\': \'FP32 model forward only, synchronized; excludes disk IO and preprocessing\', \'accuracy_drop_pp\': 100 * (metrics[\'test\'][\'accuracy\'] - metrics[\'cross_gen\'][\'accuracy\']), \'f1_drop\': metrics[\'test\'][\'f1_score\'] - metrics[\'cross_gen\'][\'f1_score\'], **{key: metadata[key] for key in (\'total_parameters\', \'trainable_parameters\', \'training_time_seconds\')}}\n        (output / \'results.json\').write_text(json.dumps(result, indent=2, allow_nan=False))\n        status.write_text(json.dumps({\'status\': \'complete\'}))\n        return result\n    except Exception as error:\n        status.write_text(json.dumps({\'status\': \'failed_not_final\', \'error\': str(error)}))\n        raise\n',
    'models/__init__.py': '',
    'models/vit/__init__.py': '',
    'models/vit/audit_progress.py': '"""Bounded image auditing and visible progress for a remote dataset mount."""\nfrom concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait\nfrom contextlib import contextmanager\nimport os\nfrom pathlib import Path\nimport threading\nimport time\n\n\nclass Progress:\n    def __init__(self, label, total=None, interval=10):\n        self.label, self.total, self.interval = label, total, interval\n        self.count = 0\n        self.detail = ""\n        self.started = time.monotonic()\n        self.stopped = threading.Event()\n\n    def message(self):\n        elapsed = time.monotonic() - self.started\n        count = f"{self.count:,}" + (f"/{self.total:,}" if self.total is not None else "")\n        return f"[{self.label}] completed {count}; elapsed {elapsed:.0f}s. {self.detail}"\n\n    def heartbeat(self):\n        while not self.stopped.wait(self.interval):\n            print(self.message(), flush=True)\n\n\n@contextmanager\ndef stage(label, total=None, interval=10):\n    state = Progress(label, total, interval)\n    print(f"[{label}] starting", flush=True)\n    thread = threading.Thread(target=state.heartbeat, daemon=True)\n    thread.start()\n    try:\n        yield state\n    except BaseException:\n        print(f"[{label}] interrupted or failed; no successful completion claimed", flush=True)\n        raise\n    else:\n        print(state.message(), flush=True)\n        print(f"[{label}] complete", flush=True)\n    finally:\n        state.stopped.set()\n        thread.join(timeout=1)\n\n\ndef discover_images(root, extensions):\n    """Do not issue a separate is_file/stat call for every image candidate.\n\n    Actual image opens/decoding remain authoritative; a broken or non-image\n    entry with an image extension fails auditing rather than being skipped.\n    """\n    root = Path(root)\n    files = []\n    with stage("Discover image paths") as progress:\n        for folder in ("Real", "Fake"):\n            stack = [root / folder]\n            while stack:\n                directory = stack.pop()\n                progress.detail = f"Listing {directory}; unchanged count means waiting on directory access."\n                with os.scandir(directory) as entries:\n                    for entry in entries:\n                        if Path(entry.name).suffix.lower() in extensions:\n                            files.append(Path(entry.path).relative_to(root).as_posix())\n                            progress.count = len(files)\n                        elif entry.is_dir(follow_symlinks=False):\n                            stack.append(Path(entry.path))\n        progress.detail = "Sorting filenames deterministically."\n        files.sort()\n    return files\n\n\ndef bounded_map(function, items, workers, progress):\n    if not isinstance(workers, int) or workers < 1:\n        raise ValueError("AUDIT_WORKERS must be a positive integer")\n    executor = ThreadPoolExecutor(max_workers=workers)\n    pending = {}\n    iterator = iter(enumerate(items))\n    results = {}\n\n    def fill():\n        while len(pending) < workers * 2:\n            try:\n                index, item = next(iterator)\n            except StopIteration:\n                break\n            pending[executor.submit(function, item)] = index\n\n    try:\n        fill()\n        while pending:\n            done, _ = wait(pending, timeout=1, return_when=FIRST_COMPLETED)\n            for future in done:\n                index = pending.pop(future)\n                results[index] = future.result()\n                progress.count += 1\n            progress.detail = (f"At most {workers * 2} queued/running tasks; "\n                               "unchanged count means workers have not completed another file.")\n            fill()\n    except BaseException:\n        for future in pending:\n            future.cancel()\n        # Python cannot cancel a thread blocked inside an OS filesystem read.\n        executor.shutdown(wait=False, cancel_futures=True)\n        raise\n    else:\n        executor.shutdown(wait=True)\n    return [results[index] for index in sorted(results)]\n\n\ndef inspect_inventory(root, files, inspect_image, workers):\n    with stage("Read, decode and hash images", total=len(files)) as progress:\n        return bounded_map(inspect_image, ((root, name) for name in files), workers, progress)\n\n\ndef verify_inventory(root, inventory, sha256_file, workers):\n    def verify(row):\n        if sha256_file(root / row["filepath"]) != row["sha256"]:\n            raise ValueError(f"Dataset content changed: {row[\'filepath\']}")\n        return None\n    with stage("Verify existing snapshot", total=len(inventory)) as progress:\n        bounded_map(verify, inventory, workers, progress)\n\n\ndef timed_split_assignment(assign_splits, inventory, seed):\n    with stage("Group duplicates and assign splits") as progress:\n        progress.detail = "CPU-only step; completed count is reported on finish, with separate grouping messages."\n        result = assign_splits(inventory, seed)\n        progress.count = len(inventory)\n        return result\n',
    'models/vit/quality_report.py': '"""ViT quality evidence from recorded history and frozen per-image predictions."""\nfrom pathlib import Path\nimport hashlib\nimport json\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (accuracy_score, average_precision_score, classification_report,\n                             cohen_kappa_score, confusion_matrix, log_loss,\n                             matthews_corrcoef, precision_recall_curve,\n                             precision_recall_fscore_support, roc_auc_score, roc_curve)\n\n\ndef calibration_bins(labels, scores, bins=10):\n    y, p = np.asarray(labels), np.asarray(scores, dtype=float)\n    indices = np.minimum((p * bins).astype(int), bins - 1)\n    return pd.DataFrame([{"bin": i, "count": int((indices == i).sum()),\n                          "mean_probability": float(p[indices == i].mean()),\n                          "observed_fake_fraction": float(y[indices == i].mean())}\n                         for i in range(bins) if (indices == i).any()])\n\n\ndef compute_metrics(labels, scores, threshold=0.5):\n    y, p = np.asarray(labels), np.asarray(scores, dtype=float)\n    if y.ndim != 1 or p.shape != y.shape or not len(y):\n        raise ValueError("Nonempty aligned label/probability vectors required")\n    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Labels must be 0/1; probabilities finite and in [0,1]")\n    if not 0 < threshold < 1:\n        raise ValueError("Threshold must be in (0,1)")\n    predicted = (p >= threshold).astype(int)\n    precision, recall, f1, _ = precision_recall_fscore_support(y, predicted, average="binary", zero_division=0)\n    cm = confusion_matrix(y, predicted, labels=[0, 1])\n    tn, fp, fn, tp = map(int, cm.ravel())\n    both = len(np.unique(y)) == 2\n    specificity = tn / (tn + fp) if tn + fp else None\n    calibration = calibration_bins(y, p)\n    ece = float((calibration["count"] * (calibration.mean_probability - calibration.observed_fake_fraction).abs()).sum() / len(y))\n    return {"accuracy": float(accuracy_score(y, predicted)),\n            "balanced_accuracy": float((specificity + recall) / 2) if both else None,\n            "precision": float(precision), "recall": float(recall), "f1_score": float(f1),\n            "roc_auc": float(roc_auc_score(y, p)) if both else None,\n            "average_precision": float(average_precision_score(y, p)) if both else None,\n            "specificity": specificity, "false_positive_rate": fp / (tn + fp) if tn + fp else None,\n            "false_negative_rate": fn / (tp + fn) if tp + fn else None,\n            "mcc": float(matthews_corrcoef(y, predicted)) if both else None,\n            "cohen_kappa": float(cohen_kappa_score(y, predicted)) if both else None,\n            "log_loss": float(log_loss(y, p, labels=[0, 1])),\n            "brier_score": float(np.mean((p - y) ** 2)), "ece_10_equal_width_bins": ece,\n            "tn": tn, "fp": fp, "fn": fn, "tp": tp, "confusion_matrix": cm.tolist(),\n            "n": len(y), "threshold": threshold}\n\n\ndef pyplot():\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    return plt\n\n\ndef save(fig, path):\n    fig.tight_layout()\n    fig.savefig(path, dpi=180, bbox_inches="tight")\n    pyplot().close(fig)\n\n\ndef learning_curves(history, output):\n    frame, output = pd.DataFrame(history), Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    plt = pyplot()\n    fig, axes = plt.subplots(2, 4, figsize=(17, 8))\n    for axis, metric, title in zip(axes.flat, ("loss", "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "lr"),\n                                    ("BCE loss", "Accuracy", "Fake precision", "Fake recall", "Fake F1", "ROC-AUC", "Average precision (PR)", "Learning rate")):\n        plotted = False\n        if metric == "lr" and "lr" in frame:\n            axis.plot(frame.epoch, frame.lr, color="#333333", marker=".")\n            plotted = True\n        elif metric != "lr":\n            for split, color in (("train", "#007f86"), ("val", "#c33d5b")):\n                key = f"{split}_{metric}"\n                if key in frame:\n                    axis.plot(frame.epoch, frame[key], label=split, color=color, marker=".")\n                    plotted = True\n            if plotted:\n                axis.legend()\n        if not plotted:\n            axis.text(0.5, 0.5, "Not recorded in this run", ha="center", transform=axis.transAxes)\n        axis.set(title=title, xlabel="Epoch")\n        if metric not in ("loss", "lr"):\n            axis.set_ylim(0, 1.02)\n        axis.grid(alpha=0.2)\n    fig.suptitle("Training and validation only; training metrics use augmented batches", fontsize=13)\n    save(fig, output)\n    if "elapsed_seconds" in frame:\n        fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n        axes[0].plot(frame.epoch, frame.elapsed_seconds / 3600)\n        axes[0].set(title="Cumulative training + validation time", xlabel="Epoch", ylabel="Hours")\n        axes[1].bar(frame.epoch, frame.elapsed_seconds.diff().fillna(frame.elapsed_seconds) / 60)\n        axes[1].set(title="Epoch duration", xlabel="Epoch", ylabel="Minutes")\n        save(fig, output.with_name("epoch_timing.png"))\n\n\ndef validate_predictions(frame):\n    required = {"model", "run_name", "filepath", "label", "probability", "predicted_label", "split", "generator"}\n    if not required.issubset(frame):\n        raise ValueError(f"Predictions missing columns: {sorted(required - set(frame))}")\n    if frame[list(required)].isna().any().any():\n        raise ValueError("Predictions contain missing required values")\n    if frame[["model", "run_name"]].drop_duplicates().shape[0] != 1:\n        raise ValueError("This report requires exactly one model/run")\n    if set(frame.split) != {"test", "cross_gen"}:\n        raise ValueError("Report requires primary and cross-generator test predictions")\n    if frame.filepath.duplicated().any():\n        raise ValueError("Repeated filepath / test split overlap")\n    if not np.array_equal(frame.predicted_label, (frame.probability >= 0.5).astype(int)):\n        raise ValueError("Saved predicted labels do not match the frozen 0.5 threshold")\n    for _, group in frame.groupby("split"):\n        compute_metrics(group.label, group.probability)\n        if set(group.label) != {0, 1}:\n            raise ValueError("Each test domain must contain real and fake controls")\n\n\ndef make_figures(predictions, output, threshold=0.5):\n    if threshold != 0.5:\n        raise ValueError("Reporting cannot retune the frozen threshold")\n    output = Path(output)\n    frame = pd.read_csv(predictions)\n    validate_predictions(frame)\n    output.mkdir(parents=True, exist_ok=True)\n    plt = pyplot()\n    parts = [(split, frame[frame.split == split]) for split in ("test", "cross_gen")]\n    results = {split: compute_metrics(group.label, group.probability) for split, group in parts}\n    rows = [{"split": split, **{k: v for k, v in result.items() if k != "confusion_matrix"}}\n            for split, result in results.items()]\n    summary = pd.DataFrame(rows)\n    summary.to_csv(output / "metrics_from_predictions.csv", index=False)\n    (output / "metrics.json").write_text(json.dumps(results, indent=2, allow_nan=False))\n    reports = {split: classification_report(group.label, group.predicted_label, labels=[0, 1],\n                                             target_names=["Real", "Fake"], output_dict=True, zero_division=0)\n               for split, group in parts}\n    (output / "classification_reports.json").write_text(json.dumps(reports, indent=2))\n    pd.DataFrame([{"split": split, "class": cls, **reports[split][cls]}\n                  for split, _ in parts for cls in ("Real", "Fake")]).to_csv(output / "per_class_metrics.csv", index=False)\n\n    fig, axes = plt.subplots(2, 2, figsize=(10, 8))\n    for col, (split, _) in enumerate(parts):\n        cm = np.array(results[split]["confusion_matrix"])\n        for row in (0, 1):\n            values = cm if row == 0 else cm / cm.sum(axis=1, keepdims=True)\n            axis = axes[row, col]\n            axis.imshow(values, cmap="Blues", vmin=0, vmax=values.max() if row == 0 else 1)\n            for y in (0, 1):\n                for x in (0, 1):\n                    text = str(cm[y, x]) if row == 0 else f"{values[y, x]:.1%}"\n                    axis.text(x, y, text, ha="center", va="center", color="white" if values[y, x] > values.max() / 2 else "black")\n            axis.set(title=f"{split}: {\'counts\' if row == 0 else \'row-normalized\'}", xlabel="Predicted", ylabel="True",\n                     xticks=[0, 1], yticks=[0, 1], xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])\n    save(fig, output / "confusion_counts_and_rates.png")\n\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    for (split, group), color in zip(parts, ("#007f86", "#c33d5b")):\n        fpr, tpr, _ = roc_curve(group.label, group.probability)\n        precision, recall, _ = precision_recall_curve(group.label, group.probability)\n        axes[0].plot(fpr, tpr, color=color, label=f"{split}: AUC {results[split][\'roc_auc\']:.3f}")\n        axes[1].plot(recall, precision, color=color, label=f"{split}: AP {results[split][\'average_precision\']:.3f}")\n        axes[1].axhline(group.label.mean(), color=color, linestyle=":", alpha=0.5)\n    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)\n    axes[0].set(title="ROC", xlabel="False positive rate", ylabel="True positive rate")\n    axes[1].set(title="Precision-recall (dotted = fake prevalence)", xlabel="Recall", ylabel="Precision")\n    for axis in axes:\n        axis.set_xlim(0, 1)\n        axis.set_ylim(0, 1.02)\n        axis.legend()\n    save(fig, output / "roc_precision_recall.png")\n\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    bins = []\n    for axis, (split, group) in zip(axes, parts):\n        table = calibration_bins(group.label, group.probability)\n        table.insert(0, "split", split)\n        bins.append(table)\n        axis.plot([0, 1], [0, 1], "k--", alpha=0.4)\n        axis.plot(table.mean_probability, table.observed_fake_fraction, "o-", color="#007f86")\n        axis.set(title=f"{split}: reliability (10 bins)", xlabel="Mean predicted fake probability",\n                 ylabel="Observed fake fraction", xlim=(0, 1), ylim=(0, 1))\n    pd.concat(bins).to_csv(output / "calibration_bins.csv", index=False)\n    save(fig, output / "calibration.png")\n\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    for axis, (split, group) in zip(axes, parts):\n        for label, name, color in ((0, "Real", "#007f86"), (1, "Fake", "#c33d5b")):\n            axis.hist(group[group.label == label].probability, bins=np.linspace(0, 1, 26), alpha=0.5, label=name, color=color)\n        axis.axvline(0.5, color="black", linestyle="--")\n        axis.set(title=split, xlabel="Predicted fake probability", ylabel="Image count")\n        axis.legend()\n    save(fig, output / "probability_distributions.png")\n\n    names = ["accuracy", "precision", "recall", "f1_score", "roc_auc", "average_precision"]\n    ax = summary.set_index("split")[names].T.plot.bar(figsize=(11, 5), rot=15, ylim=(0, 1.05), color=["#007f86", "#c33d5b"])\n    ax.set(ylabel="Score", title="Absolute performance by test domain; higher is better")\n    save(ax.figure, output / "domain_comparison.png")\n    gaps = {"accuracy_drop_pp": 100 * (results["test"]["accuracy"] - results["cross_gen"]["accuracy"]),\n            "f1_drop": results["test"]["f1_score"] - results["cross_gen"]["f1_score"]}\n    pd.DataFrame([gaps]).to_csv(output / "generalization_gaps.csv", index=False)\n    source_rows = []\n    for (split, source), group in frame.groupby(["split", "generator"]):\n        source_rows.append({"split": split, "source": source, "n": len(group),\n                            "accuracy": float((group.label == group.predicted_label).mean()),\n                            "fake_prediction_rate": float(group.predicted_label.mean()),\n                            "real_count": int((group.label == 0).sum()), "fake_count": int((group.label == 1).sum())})\n    pd.DataFrame(source_rows).to_csv(output / "source_breakdown.csv", index=False)\n    errors = frame[frame.label != frame.predicted_label].copy()\n    errors["error_type"] = np.where(errors.label == 0, "false_positive", "false_negative")\n    errors["wrong_confidence"] = np.where(errors.label == 0, errors.probability, 1 - errors.probability)\n    errors.sort_values("wrong_confidence", ascending=False).to_csv(output / "failure_cases.csv", index=False)\n    checksum = hashlib.sha256(Path(predictions).read_bytes()).hexdigest()\n    (output / "report_provenance.json").write_text(json.dumps({"predictions_sha256": checksum, "threshold": 0.5,\n        "positive_class": "fake=1", "scope": "independent ViT, two test domains", "no_new_inference": True,\n        "notes": ["Average precision is step-weighted AP, not trapezoidal PR area.",\n                  "ECE uses ten equal-width fake-probability bins and is sample/bin dependent.",\n                  "Source-only accuracy is not a balanced binary generalization score."]}, indent=2))\n    return summary\n\n\ndef failure_gallery(errors_path, root, output):\n    from PIL import Image, ImageOps\n    errors = pd.read_csv(errors_path)\n    root = Path(root).resolve()\n    selected = pd.concat([errors[(errors.split == split) & (errors.error_type == kind)].head(2)\n                          for split in ("test", "cross_gen") for kind in ("false_positive", "false_negative")])\n    if selected.empty:\n        return\n    fig, axes = pyplot().subplots(2, 4, figsize=(13, 7))\n    for axis in axes.flat:\n        axis.axis("off")\n    for axis, (_, row) in zip(axes.flat, selected.iterrows()):\n        path = (root / row.filepath).resolve()\n        if not path.is_relative_to(root):\n            raise ValueError("Image path escapes the dataset root")\n        if path.is_file():\n            with Image.open(path) as image:\n                axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))\n        else:\n            axis.text(0.5, 0.5, "Image not attached", ha="center", transform=axis.transAxes)\n        axis.set_title(f"{row.split}: {row.error_type}\\np(fake)={row.probability:.3f}", fontsize=9)\n    save(fig, Path(output) / "high_confidence_mistakes.png")\n\n\ndef render_dashboard(run_dir, evaluation_dir, output, data_root=None):\n    run_dir, evaluation_dir, output = Path(run_dir), Path(evaluation_dir), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    history = run_dir / "training_history.csv"\n    if history.exists():\n        recorded = pd.read_csv(history)\n        learning_curves(recorded, output / "training_dashboard.png")\n        metadata_path = run_dir / "run_metadata.json"\n        metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}\n        info = {"completed_epochs": int(recorded.epoch.iloc[-1]),\n                "training_time_seconds": float(recorded.elapsed_seconds.iloc[-1]) if "elapsed_seconds" in recorded else None,\n                "total_parameters": metadata.get("total_parameters"),\n                "trainable_parameters": metadata.get("trainable_parameters"),\n                "status": "complete" if (run_dir / "training_complete.json").exists() else "incomplete_or_paused"}\n        if info["trainable_parameters"] is None and "trainable_parameters" in recorded:\n            info["trainable_parameters"] = int(recorded.trainable_parameters.iloc[-1])\n        pd.DataFrame([info]).to_csv(output / "training_efficiency.csv", index=False)\n        if info["total_parameters"] is not None and info["trainable_parameters"] is not None:\n            fig, axis = pyplot().subplots(figsize=(7, 4))\n            axis.bar(["Total", "Trainable"], [info["total_parameters"] / 1e6, info["trainable_parameters"] / 1e6], color=["#007f86", "#c33d5b"])\n            axis.set(ylabel="Million parameters", title="Model size (not a detection-quality score)")\n            save(fig, output / "parameter_counts.png")\n    status = evaluation_dir / "status.json"\n    if not status.exists() or json.loads(status.read_text()).get("status") != "complete":\n        print("Training curves available if recorded. Final metrics await a completed sealed evaluation.")\n        return None\n    summary = make_figures(evaluation_dir / "predictions.csv", output)\n    result = json.loads((evaluation_dir / "results.json").read_text())\n    efficiency = []\n    for split, timing in result["timing"].items():\n        efficiency.append({"split": split, **timing,\n                           **{key: result[key] for key in ("total_parameters", "trainable_parameters", "training_time_seconds")}})\n    pd.DataFrame(efficiency).to_csv(output / "efficiency.csv", index=False)\n    table = pd.DataFrame(efficiency).set_index("split")\n    ax = table[["single_image_ms", "batch_ms_per_image"]].plot.bar(rot=0, figsize=(9, 4))\n    ax.set(ylabel="Milliseconds per image", title="Synchronized model-forward latency (no IO/preprocessing)")\n    save(ax.figure, output / "inference_latency.png")\n    if data_root is not None:\n        failure_gallery(output / "failure_cases.csv", data_root, output)\n    print("Quality dashboard:", output)\n    return summary\n',
    'models/vit/distributed.py': '"""Process-group helpers for one device or several GPUs (DDP) under torchrun.\n\ntorchrun exports RANK, LOCAL_RANK and WORLD_SIZE. Without them every helper\ndegrades to a single process, so the same trainer runs on one T4, on Kaggle\'s\ndual T4 accelerator, or on a CPU (gloo) for tests.\n"""\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\nfrom datetime import timedelta\n\nimport torch\nimport torch.distributed as dist\n\n\n@dataclass(frozen=True)\nclass Context:\n    rank: int\n    local_rank: int\n    world: int\n    device: torch.device\n\n    @property\n    def main(self):\n        return self.rank == 0\n\n\ndef setup(device_name="auto", timeout_minutes=180):\n    """Join the process group (if torchrun started several processes) and pin one device per rank.\n\n    The long timeout covers rank 0 re-hashing the training images while the other rank\n    waits at a barrier; a crashed rank is still torn down at once by torchrun.\n    """\n    world = int(os.environ.get("WORLD_SIZE", "1"))\n    rank = int(os.environ.get("RANK", "0"))\n    local = int(os.environ.get("LOCAL_RANK", "0"))\n    if device_name == "auto":\n        device_name = "cuda" if torch.cuda.is_available() else "cpu"\n    if device_name.startswith("cuda"):\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA requested but unavailable. Select Kaggle\'s GPU T4 x2 accelerator.")\n        if local >= torch.cuda.device_count():\n            raise RuntimeError(f"LOCAL_RANK={local}, but only {torch.cuda.device_count()} GPU(s) are visible.")\n        torch.cuda.set_device(local)\n        device, backend = torch.device("cuda", local), "nccl"\n    else:\n        device, backend = torch.device(device_name), "gloo"\n    if world > 1 and not dist.is_initialized():\n        dist.init_process_group(backend=backend, timeout=timedelta(minutes=timeout_minutes))\n    return Context(rank=rank, local_rank=local, world=world, device=device)\n\n\ndef active(ctx):\n    return ctx.world > 1 and dist.is_available() and dist.is_initialized()\n\n\ndef barrier(ctx):\n    if not active(ctx):\n        return\n    if ctx.device.type == "cuda":\n        dist.barrier(device_ids=[ctx.device.index])\n    else:\n        dist.barrier()\n\n\ndef all_gather_object(ctx, value):\n    """Every rank receives the list of every rank\'s (picklable) value, ordered by rank."""\n    if not active(ctx):\n        return [value]\n    gathered = [None] * ctx.world\n    dist.all_gather_object(gathered, value)\n    return gathered\n\n\ndef broadcast_object(ctx, value):\n    """Rank 0\'s value on every rank; keeps pause/stop decisions identical so no rank waits forever."""\n    if not active(ctx):\n        return value\n    box = [value if ctx.main else None]\n    dist.broadcast_object_list(box, src=0)\n    return box[0]\n\n\ndef cleanup(ctx):\n    if active(ctx):\n        dist.destroy_process_group()\n',
    'models/vit/launch_train.py': '"""torchrun entry point, one process per GPU:\n\n    python -m torch.distributed.run --standalone --nproc_per_node=2 -m models.vit.launch_train CONFIG.json\n"""\nimport json\nimport sys\n\nfrom models.vit.wish_standalone import train_or_resume\n\n\ndef main():\n    if len(sys.argv) != 2:\n        raise SystemExit("usage: -m models.vit.launch_train CONFIG.json")\n    with open(sys.argv[1]) as handle:\n        train_or_resume(json.load(handle))\n\n\nif __name__ == "__main__":\n    main()\n',
    'models/vit/nccl_check.py': '"""GPU to GPU communication preflight, run under torchrun before any training.\n\nAll reduces a buffer the size of ViT-B/16\'s fp32 gradients, checks the result and\ntimes it. A broken PCIe peer to peer path shows up here within seconds (the notebook\nthen retries with NCCL_P2P_DISABLE=1) instead of as a hang in the middle of training.\nWithout CUDA it runs the same check over gloo on the CPU, which is only for testing.\n"""\nimport time\n\nimport torch\nimport torch.distributed as dist\n\nfrom models.vit import distributed as dist_utils\n\nVIT_B16_PARAMETERS = 86_000_000\n\n\ndef main():\n    cuda = torch.cuda.is_available()\n    ctx = dist_utils.setup("cuda" if cuda else "cpu", timeout_minutes=3)\n    try:\n        if ctx.world < 2:\n            print("Single process: no inter GPU communication to check.", flush=True)\n            return\n        elements = VIT_B16_PARAMETERS if cuda else 1_000_000\n        buffer = torch.full((elements,), float(ctx.rank + 1), device=ctx.device)\n        dist.all_reduce(buffer)\n        if cuda:\n            torch.cuda.synchronize(ctx.device)\n        expected = ctx.world * (ctx.world + 1) / 2\n        if not (torch.all(buffer[:1024] == expected) and torch.all(buffer[-1024:] == expected)):\n            raise RuntimeError("All reduce returned wrong values.")\n        repeats = 5\n        start = time.perf_counter()\n        for _ in range(repeats):\n            dist.all_reduce(buffer)\n        if cuda:\n            torch.cuda.synchronize(ctx.device)\n        milliseconds = 1000 * (time.perf_counter() - start) / repeats\n        names = dist_utils.all_gather_object(ctx, torch.cuda.get_device_name(ctx.device) if cuda else "cpu")\n        if ctx.main:\n            backend = "NCCL" if cuda else "gloo (CPU test mode)"\n            print(f"{backend} OK across {ctx.world} devices {names}. All reduce of {elements * 4 / 2**20:.0f} MiB "\n                  f"(a full ViT-B/16 fp32 gradient on GPU): {milliseconds:.0f} ms; DDP overlaps it with backward.",\n                  flush=True)\n    finally:\n        dist_utils.cleanup(ctx)\n\n\nif __name__ == "__main__":\n    main()\n',
    'models/vit/presize_cache.py': '"""Exact 224x224 decode cache: pay image decoding and resizing once per session.\n\nThe first training and validation transform is a deterministic Resize to 224x224.\nStoring its uint8 output and applying the remaining (random) transforms to it gives\npixel-identical tensors, so the cache changes speed only, never the experiment.\nIt lives in local scratch space (never /kaggle/working, so it is not published)\nand is rebuilt in each Kaggle session. Training falls back to disk reads when the\ncache is absent, stale or does not match the model\'s preprocessing.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nfrom torchvision import transforms as T\n\nfrom models.vit.audit_progress import bounded_map, stage\nfrom models.vit.manifest import image_path, load_audited_manifest\n\nFORMAT = "wish_presize_v1"\nENVIRONMENT_VARIABLE = "WISH_PRESIZE_CACHE"\n\n\ndef resize_op(preprocessing):\n    size = preprocessing["image_size"]\n    return T.Resize((size, size), interpolation=T.InterpolationMode(preprocessing["interpolation"]))\n\n\ndef signature(manifest_sha256, filepaths, preprocessing):\n    return {"format": FORMAT, "manifest_sha256": manifest_sha256,\n            "filepaths_sha256": hashlib.sha256("\\n".join(filepaths).encode()).hexdigest(),\n            "image_size": preprocessing["image_size"], "interpolation": preprocessing["interpolation"],\n            "count": len(filepaths)}\n\n\ndef build(manifest, root, directory, preprocessing, splits=("train", "val"), workers=4, headroom_bytes=2 << 30):\n    """Create (or verify and reuse) the cache. Returns its directory, or None when there is no room."""\n    rows, audit = load_audited_manifest(manifest, root, verify_splits=())\n    filepaths = sorted(row["filepath"] for row in rows if row["split"] in splits)\n    expected = signature(audit["manifest_sha256"], filepaths, preprocessing)\n    directory = Path(directory)\n    size = preprocessing["image_size"]\n    shape = (len(filepaths), size, size, 3)\n    data = directory / "images.u8"\n    meta = directory / "cache.json"\n    if meta.is_file() and data.is_file() and json.loads(meta.read_text()) == expected \\\n            and data.stat().st_size == int(np.prod(shape)):\n        print(f"Reusing presize cache: {len(filepaths):,} images in {directory}", flush=True)\n        return directory\n    needed = int(np.prod(shape))\n    directory.mkdir(parents=True, exist_ok=True)\n    meta.unlink(missing_ok=True)\n    data.unlink(missing_ok=True)\n    free = shutil.disk_usage(directory).free\n    if free < needed + headroom_bytes:\n        print(f"Presize cache skipped: needs {needed / 2**30:.1f} GiB, only {free / 2**30:.1f} GiB free "\n              f"in {directory}. Training reads images from disk instead.", flush=True)\n        return None\n    array = np.memmap(data, dtype=np.uint8, mode="w+", shape=shape)\n    resize = resize_op(preprocessing)\n\n    def store(item):\n        index, relative = item\n        with Image.open(image_path(root, relative)) as image:\n            array[index] = np.asarray(resize(ImageOps.exif_transpose(image).convert("RGB")), dtype=np.uint8)\n        return None\n\n    print(f"Building exact {size}x{size} presize cache for {len(filepaths):,} train/val images "\n          f"({needed / 2**30:.1f} GiB in {directory}).", flush=True)\n    with stage("Decode and resize once", total=len(filepaths)) as progress:\n        bounded_map(store, list(enumerate(filepaths)), workers, progress)\n    array.flush()\n    del array\n    (directory / "index.json").write_text(json.dumps(filepaths))\n    meta.write_text(json.dumps(expected, indent=2))  # written last: marks the cache complete\n    return directory\n\n\nclass PresizeCache:\n    """Read-only view shared by DataLoader workers; the memmap opens lazily inside each worker."""\n\n    def __init__(self, directory, preprocessing, manifest_sha256):\n        directory = Path(directory)\n        meta = json.loads((directory / "cache.json").read_text())\n        filepaths = json.loads((directory / "index.json").read_text())\n        if meta != signature(manifest_sha256, filepaths, preprocessing):\n            raise ValueError("Presize cache does not match this manifest or preprocessing.")\n        size = preprocessing["image_size"]\n        self.path = directory / "images.u8"\n        self.shape = (len(filepaths), size, size, 3)\n        self.index = {relative: position for position, relative in enumerate(filepaths)}\n        self._array = None\n\n    def __getstate__(self):\n        state = dict(self.__dict__)\n        state["_array"] = None  # never pickle gigabytes into worker processes\n        return state\n\n    def get(self, relative):\n        position = self.index.get(relative)\n        if position is None:\n            return None\n        if self._array is None:\n            self._array = np.memmap(self.path, dtype=np.uint8, mode="r", shape=self.shape)\n        return Image.fromarray(np.array(self._array[position]))\n\n\ndef open_cache(preprocessing, manifest_sha256):\n    directory = os.environ.get(ENVIRONMENT_VARIABLE)\n    if not directory:\n        return None\n    try:\n        return PresizeCache(directory, preprocessing, manifest_sha256)\n    except (OSError, ValueError, KeyError, json.JSONDecodeError) as error:\n        print(f"Presize cache ignored ({error}); reading images from disk.", flush=True)\n        return None\n',
}
CODE_ROOT = OUTPUT_ROOT / "code"
if any(name == "models" or name.startswith("models.") for name in sys.modules):
    loaded = sys.modules.get("models")
    location = getattr(loaded, "__file__", None)
    if not location or not Path(location).resolve().is_relative_to(CODE_ROOT.resolve()):
        raise RuntimeError("Restart this notebook's session: a different models package is already imported.")
for relative, source in SOURCE_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.read_text() != source:
        raise RuntimeError("Runtime differs from a previous run; choose a new OUTPUT_ROOT.")
    target.write_text(source)
bundle_digest = hashlib.sha256(json.dumps(SOURCE_FILES, sort_keys=True).encode()).hexdigest()
(OUTPUT_ROOT / "runtime_sha256.txt").write_text(bundle_digest)
os.chdir(CODE_ROOT)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
print("Self-contained runtime ready:", bundle_digest[:16])

## Locate Wish images
The attached Wish dataset is preferred. A KaggleHub fallback uses the same dataset handle.
An unknown numeric version is not invented: every image is hashed and the exact content snapshot
is recorded in the audit. An explicit KAGGLE_VERSION instead requests that version through KaggleHub.

In [ ]:
from models.vit.wish_standalone import find_root, prepare
DATASET_HANDLE = "wish096/realvsfake-81k-by-wish"
if KAGGLE_VERSION is not None:
    assert isinstance(KAGGLE_VERSION, int) and not isinstance(KAGGLE_VERSION, bool) and KAGGLE_VERSION > 0
    assert DATASET_INPUT is None, "Choose either an explicit input folder or a version-pinned download."
    import kagglehub
    DATASET_DIR = Path(kagglehub.dataset_download(f"{DATASET_HANDLE}/versions/{KAGGLE_VERSION}"))
elif DATASET_INPUT:
    DATASET_DIR = Path(DATASET_INPUT)
else:
    candidates = [Path("/kaggle/input/datasets/wish096/realvsfake-81k-by-wish"),
                  Path("/kaggle/input/realvsfake-81k-by-wish")]
    attached = [path for path in candidates if path.is_dir()]
    if len(attached) > 1:
        raise ValueError("Multiple Wish mounts found; set DATASET_INPUT to the desired one.")
    if attached:
        DATASET_DIR = attached[0]
    else:
        import kagglehub
        DATASET_DIR = Path(kagglehub.dataset_download(DATASET_HANDLE))
DATA_ROOT = find_root(DATASET_DIR)
PROVENANCE = {"handle": DATASET_HANDLE, "requested_version": KAGGLE_VERSION,
              "numeric_version_status": "explicitly_requested" if KAGGLE_VERSION else "unreported_content_fingerprinted"}
print("Images:", DATA_ROOT)
print("Provenance:", PROVENANCE)

## Build and validate the independent splits
This reads the complete collection. Duration depends on dataset storage; progress prints every 10 seconds. It does not run a model on test images.

- Real=0, fake=1. Filename prefixes establish FFHQ, CelebA, StyleGAN and Stable Diffusion provenance.
- Exclude the unspecified AI subset. Stop on unreadable images, unknown filenames or conflicting exact-duplicate labels.
- Remove exact duplicate copies before splitting, with an exclusion log. Group dHash-near images before splitting;
  conservatively exclude groups mixing real/fake or StyleGAN/Stable Diffusion, also logged.
- Reserve Stable Diffusion and approximately matched real controls for cross-generator testing.
- Split the remaining real and StyleGAN groups approximately 70/15/15 into train/validation/primary test,
  stratifying by source where possible. Group integrity takes priority over exact counts.
- Face identity metadata is unavailable. Perceptual hashing does not establish identity independence.

The generated CSVs and audit are the authoritative split. Never replace them after training starts.

In [ ]:
AUDIT_DIR = OUTPUT_ROOT / "data_audit"
try:
    audit = prepare(DATA_ROOT, AUDIT_DIR, seed=SEED, provenance=PROVENANCE, workers=AUDIT_WORKERS)
except ValueError as error:
    # Splits restored from an older output that were made with other settings (for example another
    # dataset version) are set aside, but only while no run of this experiment depends on them.
    if "settings changed" not in str(error) or (OUTPUT_ROOT / "runs" / RUN_NAME).exists():
        raise
    unused = OUTPUT_ROOT / f"data_audit_restored_unused_{int(time.time())}"
    AUDIT_DIR.rename(unused)
    print(f"Restored splits used different settings; moved to {unused.name} and building new splits.")
    audit = prepare(DATA_ROOT, AUDIT_DIR, seed=SEED, provenance=PROVENANCE, workers=AUDIT_WORKERS)
manifest = pd.read_csv(AUDIT_DIR / "manifest.csv")
display(pd.crosstab(manifest["split"], manifest["source"]))
print("Manifest SHA256:", audit["manifest_sha256"])
print("Dataset snapshot:", audit["dataset_version"])
print("Excluded with recorded reasons:", audit["exclusions"])
print("Detected near pairs grouped/excluded:", audit["detected_near_pairs_before_grouping"])
print("Identity independence has not been verified.")

## ViT configuration and dual GPU launch
ViT-B/16 uses 224x224 RGB input and its checkpoint specific normalization, a one logit head,
BCEWithLogitsLoss, AdamW at 3e-5, weight decay 0.01, 10% linear warmup and decay, and gradient clipping 1.0.
Each optimizer step uses 32 images: 16 on each T4, gradients averaged across both, no accumulation.
CUDA mixed precision is enabled. Early stopping uses validation loss with patience 6.
Neither test set is loaded by the trainer.

`launch()` starts one training process per GPU with `torchrun`. Only the first process writes files;
decisions to pause or stop are broadcast so both processes always agree. The per GPU random seeds differ
(independent dropout and augmentation noise per GPU), and each process saves its own random state, so a
resumed run continues exactly where it stopped. The configuration records `N_GPUS` and the runtime code hash,
so a resume with a different GPU count or changed code is refused rather than silently mixed.

In [ ]:
CFG = {
    "model_name": "vit_b16", "module": "models.vit.model", "run_name": RUN_NAME,
    "seed": SEED, "device": "cuda", "runtime_sha256": bundle_digest,
    "distributed": {"world_size": N_GPUS},
    "data": {"root": str(DATA_ROOT), "manifest": str(AUDIT_DIR / "manifest.csv"),
             "near_duplicate_review": None, "batch_size": MICROBATCH_PER_GPU, "num_workers": NUM_WORKERS_PER_GPU},
    "model": {"backbone": "vit_base_patch16_224.augreg_in21k_ft_in1k", "pretrained": True,
              "num_classes": 1, "dropout": 0.1, "use_frequency_hybrid": False},
    "train": {"epochs": EPOCHS, "early_stopping_patience": 6, "optimizer": "adamw",
              "lr": 3e-5, "weight_decay": 0.01, "warmup_steps_pct": 0.1,
              "effective_batch_size": EFFECTIVE_BATCH, "grad_clip_norm": 1.0,
              "mixed_precision": True, "progress_every": 100},
    "output": {"results_dir": str(OUTPUT_ROOT / "runs")},
}
CONFIG_PATH = OUTPUT_ROOT / f"{RUN_NAME}.json"
if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != CFG:
    raise ValueError("Run configuration or runtime code changed. Choose a new RUN_NAME; do not alter an in progress run.")
CONFIG_PATH.write_text(json.dumps(CFG, indent=2))
RUN_DIR = Path(CFG["output"]["results_dir"]) / RUN_NAME
print(json.dumps(CFG, indent=2))


def torchrun(module, *args, timeout=None, env=None):
    """One process per GPU. torchrun stops every process if any one of them fails."""
    env = dict(os.environ if env is None else env)
    env.setdefault("OMP_NUM_THREADS", "1")  # 4 CPU cores are shared by 2 trainers and their data loaders
    run("-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={N_GPUS}", "-m", module, *args,
        timeout=timeout, env=env)


def launch(config_path, timeout=None):
    torchrun("models.vit.launch_train", config_path, timeout=timeout)

## Exact presize cache
Kaggle's T4 x2 machine has only 4 CPU cores for two GPUs, and decoding full size face JPEGs every epoch
can starve them. This cell decodes and resizes each train and validation image to 224x224 once, into
`/tmp` (local scratch space, not published). The first training transform is exactly that deterministic
resize, so augmenting the cached pixels gives tensors identical to reading the original files; this was
verified to be bit exact. It is rebuilt in every new session, skipped when disk space is short or when no
training is pending, and ignored by the trainer if it does not match the manifest and preprocessing.

In [ ]:
from models.vit import presize_cache
os.environ.pop(presize_cache.ENVIRONMENT_VARIABLE, None)
TRAINING_PENDING = (RUN_FULL_TRAINING and not (RUN_DIR / "training_complete.json").is_file()
                    and not (OUTPUT_ROOT / "sealed_evaluation/frozen.json").exists())
if not PRESIZE_CACHE:
    print("Presize cache disabled: training decodes every image from disk each epoch.")
elif not TRAINING_PENDING:
    print("No training pending in this session; presize cache not needed.")
else:
    from models.vit.runtime import build_model, preprocessing_for
    PREPROCESSING = preprocessing_for(build_model(CFG, pretrained=False), CFG)  # the transform needs no weights
    started = time.perf_counter()
    cache_dir = presize_cache.build(AUDIT_DIR / "manifest.csv", DATA_ROOT, PRESIZE_CACHE_DIR, PREPROCESSING,
                                    workers=AUDIT_WORKERS)
    if cache_dir:
        os.environ[presize_cache.ENVIRONMENT_VARIABLE] = str(cache_dir)
        print(f"Presize cache ready after {time.perf_counter() - started:.0f} s; both training processes read it.")

## GPU communication check and smoke test
First a few seconds of NCCL all reduce between the two GPUs, verified and timed. If it fails or stalls,
it is retried with `NCCL_P2P_DISABLE=1` (traffic goes through host memory instead of direct PCIe) and the
working setting is kept for training.

The smoke test then trains one epoch on at most 32 training and 32 validation images across both GPUs.
It downloads the pretrained weights, checks DDP forward and backward and saves a separate smoke checkpoint.
It does not measure detector accuracy. Smoke weights are never reused for the full experiment, and a completed
identical smoke run is reused on rerun.

In [ ]:
def nccl_preflight():
    """Check GPU to GPU communication in seconds, before training could hang on it."""
    if N_GPUS < 2:
        print("One GPU: no inter GPU communication to check.")
        return
    for name, extra in (("default settings", {}), ("NCCL_P2P_DISABLE=1", {"NCCL_P2P_DISABLE": "1"})):
        try:
            torchrun("models.vit.nccl_check", timeout=300, env=dict(os.environ, **extra))
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as failure:
            print(f"GPU communication check with {name} failed ({type(failure).__name__}).")
            continue
        os.environ.update(extra)  # training inherits the setting that worked
        print("GPU communication for training:", name)
        return
    raise RuntimeError("The GPUs cannot communicate over NCCL. Restart the session; if it persists, "
                       "set N_GPUS = 1 and a new RUN_NAME.")


nccl_preflight()
SMOKE_CFG = copy.deepcopy(CFG)
SMOKE_CFG["run_name"] += "_smoke"
SMOKE_CFG["smoke"] = True
SMOKE_CFG["train"]["epochs"] = 1
SMOKE_PATH = OUTPUT_ROOT / f"{RUN_NAME}_smoke.json"
SMOKE_PATH.write_text(json.dumps(SMOKE_CFG, indent=2))
launch(SMOKE_PATH, timeout=3600)
print("Real data smoke test completed on", N_GPUS, "GPU(s). Full training starts from fresh pretrained weights.")

## Full training and resume
The next cell performs the real run on both GPUs. Each epoch writes best and last checkpoints, history,
configuration and environment details, and logs training images per second and the fraction of step time
spent waiting for data. Rerunning resumes from the last **completed epoch**; partial epoch work is repeated.
Resume preserves every GPU's random state, the optimizer, scaler and scheduler. A completed run is not trained again.

For a later session, attach the saved output as input; `RESTORE_FROM = "auto"` locates it. Keep the same output
path, dataset content, `RUN_NAME`, `N_GPUS` and settings. Different Kaggle mount paths require deliberate config
relocation; the trainer refuses a mismatch rather than silently start another experiment.
If a T4 runs out of memory, set `MICROBATCH_PER_GPU = 8` (two accumulation steps keep 32 per step) and
choose a new `RUN_NAME` before a fresh run. A hard timeout stops a hung run with time left to publish its
finished epochs.

In [ ]:
if RUN_FULL_TRAINING:
    if (OUTPUT_ROOT / "sealed_evaluation/frozen.json").exists():
        print("Experiment already sealed. Training is disabled; inspect saved results.")
    else:
        spent = time.perf_counter() - NOTEBOOK_START
        remaining = SESSION_LIMIT_SECONDS - SESSION_RESERVE_SECONDS - spent
        os.environ["WISH_TIME_BUDGET_SECONDS"] = str(max(remaining, 0.0))
        print(f"Session used so far: {spent / 3600:.2f} h. Training budget now: {remaining / 3600:.2f} h.")
        if remaining <= 0:
            print("No safe training time is left in this session. Save this version, attach its output and continue.")
        else:
            (OUTPUT_ROOT / "training_error.json").unlink(missing_ok=True)
            try:
                # The trainer's own time guard normally stops well before this; the hard limit only
                # catches a hung run and still leaves time to publish the finished epochs.
                launch(CONFIG_PATH, timeout=remaining + SESSION_RESERVE_SECONDS / 3)
            except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as failure:
                # Re-raising fails the saved version, which would discard the completed epochs.
                (OUTPUT_ROOT / "training_error.json").write_text(json.dumps(
                    {"error": type(failure).__name__, "returncode": getattr(failure, "returncode", None),
                     "run_name": RUN_NAME, "resume_state_present": (RUN_DIR / "checkpoints/last.pt").is_file()},
                    indent=2))
                print("TRAINING FAILED OR TIMED OUT. Read the log above before starting the next run.")
                if not (RUN_DIR / "checkpoints/last.pt").is_file():
                    raise
else:
    print("Full training disabled. Set RUN_FULL_TRAINING=True when ready.")
from IPython.display import Image as DisplayImage, display
if (RUN_DIR / "training_history.csv").exists():
    history = pd.read_csv(RUN_DIR / "training_history.csv")
    display(history.tail())
    if "train_images_per_second" in history and history["train_images_per_second"].notna().any():
        speed = history["train_images_per_second"].dropna()
        wait = history["train_data_wait_fraction"].dropna()
        gpus = int(history["world_size"].dropna().iloc[-1]) if "world_size" in history else N_GPUS
        print(f"Training throughput on {gpus} GPU(s): median {speed.median():.0f} images/s "
              f"({speed.median() / gpus:.0f} per GPU); median data wait {wait.median():.0%} of step time.")
        if len(wait) and wait.median() > 0.25:
            print("The input pipeline, not the GPUs, limits speed here. Check that the presize cache was built.")
    if (RUN_DIR / "learning_curves.png").exists():
        display(DisplayImage(filename=str(RUN_DIR / "learning_curves.png")))
print("Best weights:", RUN_DIR / "checkpoints/best_model.pt")
print("Resume state:", RUN_DIR / "checkpoints/last.pt")
for name in ("training_paused.json", "training_complete.json"):
    if (RUN_DIR / name).is_file():
        print(name, (RUN_DIR / name).read_text())
if (OUTPUT_ROOT / "training_error.json").is_file():
    print("Training error recorded this run:", (OUTPUT_ROOT / "training_error.json").read_text())

## Optional sealed evaluation: ViT only
Keep the flag false until all validation-based decisions and planned experiments are complete.
Then set READY_FOR_FINAL_EVALUATION=True and run this cell once. It freezes the best checkpoint hash,
manifest and threshold 0.5 before evaluating primary and cross-generator tests. No other model is required.
Never tune settings from these test results. A rerun displays saved results instead of repeating inference.

In [ ]:
EVALUATION_DIR = OUTPUT_ROOT / "sealed_evaluation"
if not READY_FOR_FINAL_EVALUATION:
    print("Tests remain sealed. Training does not require evaluation.")
elif (EVALUATION_DIR / "status.json").exists():
    state = json.loads((EVALUATION_DIR / "status.json").read_text())
    assert state["status"] == "complete", "An evaluation was interrupted; inspect its status before any retry."
    print("Using saved evaluation; no repeated test inference.")
elif not (RUN_DIR / "training_complete.json").is_file():
    print("Training chunk finished, but full training is incomplete; tests stay sealed.")
else:
    run("-u", "-c", "import json,sys; from models.vit.wish_standalone import evaluate_vit; "
        "evaluate_vit(json.load(open(sys.argv[1])),sys.argv[2],sys.argv[3])",
        CONFIG_PATH, RUN_DIR / "checkpoints/best_model.pt", EVALUATION_DIR)

if (EVALUATION_DIR / "results.json").exists():
    results = json.loads((EVALUATION_DIR / "results.json").read_text())
    display(pd.DataFrame(results["metrics"]).T)
    print("Accuracy drop (percentage points):", results["accuracy_drop_pp"])
    print("F1 drop:", results["f1_drop"])
    print("Independent ViT results only; not a four-model comparison.")
    from IPython.display import Image as DisplayImage, display
    display(DisplayImage(filename=str(EVALUATION_DIR / "figures/confusion_matrices_side_by_side.png")))

## Saved evidence and limitations
All artifacts live in `/kaggle/working/wish_vit_ddp_v4`:

- `data_audit`: content inventory, hashes, four split CSVs, exclusion reasons and perceptual candidates.
- `runs/<RUN_NAME>`: best weights, resumable optimizer and per GPU random state, history (including throughput
  and data wait per epoch), learning curves, config and environment (GPU names and count).
- `sealed_evaluation` (only when enabled): frozen protocol, per image probabilities, accuracy/precision/recall/F1/AUC,
  ROC curves, side by side confusion matrices, high confidence FP/FN CSV, parameter counts and synchronized latency.
- `gradio_app`: the app script, the example images it shows, and its self test results.

Preserve the complete output, including manifests, before ending the session. Do not publish private team material.
Share the manifest with A/B only if they agree to adopt these new splits. Prior runs on other splits are not comparable.
Record duplicate exclusions, approximate source proportions, source and compression confounding, unequal pretraining,
unknown identity overlap and single seed limitations. ViT superiority is a hypothesis, not a guaranteed result.

Sources: [Wish dataset](https://www.kaggle.com/datasets/wish096/realvsfake-81k-by-wish),
[official pretrained model card](https://huggingface.co/timm/vit_base_patch16_224.augreg_in21k_ft_in1k),
[PyTorch DDP](https://docs.pytorch.org/docs/stable/notes/ddp.html),
[KaggleHub API](https://github.com/Kaggle/kagglehub).

## Actual ViT quality dashboard
Every number below comes from saved history or frozen predictions, never example values.
Per-epoch precision/recall/F1/AP/AUC use fake=1. Training metrics use augmented batches
while weights are changing; validation metrics use deterministic transforms. Final test
metrics are separate and require explicit sealed evaluation after training completes.

The report includes class-wise metrics, specificity/FPR/FNR, balanced accuracy, MCC,
Cohen's kappa, log loss, Brier score and 10-bin probability ECE. AP is average precision,
not trapezoidal PR area. ROC/AP are undefined for a single-class source subset, so the
source breakdown does not invent them. Calibration is diagnostic only; no test-set
calibration fitting or threshold optimization is performed.

Parameter counts measure size, not detection quality. Report absolute cross-generator
performance as well as its drop. These are one-run results, not evidence of universal ViT superiority.

In [ ]:
from models.vit.quality_report import render_dashboard
QUALITY_DIR = OUTPUT_ROOT / "quality_dashboard"
summary = render_dashboard(RUN_DIR, OUTPUT_ROOT / "sealed_evaluation", QUALITY_DIR, DATA_ROOT)
if summary is not None:
    display(summary)
for table in ("training_efficiency.csv", "per_class_metrics.csv", "generalization_gaps.csv", "efficiency.csv", "source_breakdown.csv"):
    if (QUALITY_DIR / table).exists():
        print(table)
        display(pd.read_csv(QUALITY_DIR / table))
from IPython.display import Image as DisplayImage, display
for name in ("training_dashboard.png", "epoch_timing.png", "parameter_counts.png", "confusion_counts_and_rates.png",
             "roc_precision_recall.png", "domain_comparison.png", "calibration.png",
             "probability_distributions.png", "inference_latency.png", "high_confidence_mistakes.png"):
    if (QUALITY_DIR / name).exists():
        display(DisplayImage(filename=str(QUALITY_DIR / name)))

All charts are exported as PNG and all tables as CSV/JSON in `quality_dashboard`.
Missing metrics from older histories are marked unavailable, never reconstructed from guesses.
Sources: [scikit-learn metrics](https://scikit-learn.org/stable/modules/model_evaluation.html),
[calibration](https://scikit-learn.org/stable/modules/calibration.html),
[Kaggle saved runs](https://www.kaggle.com/docs/notebooks).

## Gradio app: real or fake?
The final output of this notebook is a Gradio app around the trained detector, plus an automatic test of
that app on actual real and fake images.

- It loads `best_model.pt` (the checkpoint chosen by validation loss) with the preprocessing recorded inside
  it, and applies the frozen decision rule of the sealed evaluation: **FAKE** when P(fake) ≥ 0.5,
  otherwise **REAL (not fake)**.
- Two tabs: one image with its verdict and probabilities, or many images at once as a table.
- **Self test:** the notebook starts the app and uploads real (FFHQ, CelebA) and fake (StyleGAN) dataset
  images to it over HTTP with `gradio_client`, the same route a browser upload takes. For every image it
  checks that the app's answer equals the model called directly, and it reports how many verdicts are
  correct for real images, for fake images and per source. While the tests are sealed it uses validation
  images (they already guided checkpoint selection, so the accuracy is slightly optimistic); after the
  sealed evaluation it uses never seen test images plus the held out Stable Diffusion fakes.
- Optional: set `OWN_TEST_IMAGES` in the first cell to a folder of your own images (for example an attached
  dataset with `real/` and `fake/` subfolders). They are added to the self test; images outside such folders
  are predicted but not scored.
- Results go to `gradio_app/self_test_results.csv`, `self_test_summary.json` and `self_test_grid.png`.

In an interactive session the app keeps running with a public `gradio.live` link. In a saved run it is
tested and then shut down, because the session ends with the run; any Gradio problem is logged to
`gradio_app/gradio_error.txt` and never fails the version.

**Only want the app for an already trained model?** Start an interactive session with this notebook's output
attached, run the settings cell, the included runtime cell and the "Locate Wish images" cell, then the cells
of this section. Nothing is retrained.

In [ ]:
import traceback
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
AUDIT_DIR = OUTPUT_ROOT / "data_audit"  # derived again so this section also works after only the first cells
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_NAME
GRADIO_DIR = OUTPUT_ROOT / "gradio_app"
GRADIO_DIR.mkdir(parents=True, exist_ok=True)
GRADIO_APP_PATH = GRADIO_DIR / "wish_gradio_app.py"
GRADIO_ERROR_LOG = GRADIO_DIR / "gradio_error.txt"


def gradio_step(name, function, *args, **kwargs):
    """In a saved run a failure here is logged, not raised: a failed version publishes no output at all,
    which would also lose the trained model. Interactive sessions raise so the error is visible."""
    try:
        return function(*args, **kwargs)
    except Exception:
        report = f"{name} failed:\n{traceback.format_exc()}"
        with GRADIO_ERROR_LOG.open("a") as handle:
            handle.write(report + "\n")
        print(report)
        if KAGGLE_RUN_TYPE != "Batch":
            raise
        print("Saved run continues, so the trained model and all results are still published.")
        return None


def gradio_version():
    try:
        import gradio, gradio_client
    except ImportError:
        return None
    return gradio.__version__ if int(gradio.__version__.split(".")[0]) >= 5 else None


def install_gradio():
    if gradio_version():
        print("Gradio", gradio_version(), "already available.")
        return
    from importlib import metadata
    # Pin what training depends on, so pip picks a Gradio release compatible with this environment
    # instead of replacing Kaggle's CUDA build of torch or other core packages.
    names = ("torch", "torchvision", "timm", "numpy", "pillow", "pandas", "scikit-learn",
             "matplotlib", "huggingface-hub")
    pins = {}
    for name in names:
        try:
            pins[name] = f"{name}=={metadata.version(name)}"
        except metadata.PackageNotFoundError:
            pass
    constraints = GRADIO_DIR / "pip_constraints.txt"
    core = ("torch", "torchvision", "timm", "numpy")
    for label, kept in (("strict", list(pins)), ("core only", [name for name in core if name in pins])):
        constraints.write_text("\n".join(pins[name] for name in kept) + "\n")
        try:
            run("-m", "pip", "install", "--quiet", "gradio>=5,<7", "-c", constraints, timeout=1200)
            break
        except subprocess.CalledProcessError:
            print(f"pip could not install Gradio with {label} pins; trying fewer pins.")
    importlib.invalidate_caches()
    if not gradio_version():
        raise RuntimeError("Gradio installation failed. If an import error mentions an already loaded package, "
                           "restart the kernel and run the settings, runtime, dataset and Gradio cells again.")
    print("Installed Gradio", gradio_version())


if RUN_GRADIO_APP:
    gradio_step("Gradio installation", install_gradio)
else:
    print("RUN_GRADIO_APP is False: the app script is still written below, but not installed or started.")

In [ ]:
%%writefile {GRADIO_APP_PATH}
"""Gradio demo and self test for the Wish ViT-B/16 real vs fake face detector.

Loads the validation selected checkpoint (best_model.pt) with the checkpoint's own
recorded preprocessing, and applies the frozen 0.5 threshold on P(fake) with
real=0, fake=1, exactly like the sealed evaluation. Standalone use:

    python wish_gradio_app.py --run-dir <OUTPUT_ROOT>/runs/<RUN_NAME> --code-dir <OUTPUT_ROOT>/code

Only four files of the run folder are needed: checkpoints/best_model.pt, run_config_used.json,
training_history.csv and (when training finished) training_complete.json.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from PIL import Image

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


class Detector:
    """One loaded checkpoint plus the exact inference contract used in training and evaluation."""

    def __init__(self, run_dir, checkpoint_name="best_model.pt", device=None):
        from models.vit.runtime import load_checkpoint
        self.run_dir = Path(run_dir)
        self.cfg = json.loads((self.run_dir / "run_config_used.json").read_text())
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.checkpoint = self.run_dir / "checkpoints" / checkpoint_name
        self.model, self.metadata = load_checkpoint(self.cfg, self.checkpoint, self.device)
        if self.metadata.get("smoke"):
            raise ValueError("Refusing a smoke test checkpoint: it was trained on 32 images and means nothing.")
        self.threshold = float(self.metadata["threshold"])
        self.status = self._status()

    def _status(self):
        history_path = self.run_dir / "training_history.csv"
        history = pd.read_csv(history_path) if history_path.is_file() else pd.DataFrame()
        best_epoch = self.metadata.get("best_epoch")
        best = history[history.epoch == best_epoch] if best_epoch is not None and "epoch" in history else pd.DataFrame()
        pick = lambda column: float(best[column].iloc[0]) if column in best and len(best) else None
        return {"run_name": self.metadata.get("run_name"), "best_epoch": best_epoch,
                "epochs_recorded": int(history.epoch.max()) if len(history) else 0,
                "planned_epochs": self.cfg["train"]["epochs"],
                "training_complete": (self.run_dir / "training_complete.json").is_file(),
                "val_accuracy_at_best": pick("val_accuracy"), "val_roc_auc_at_best": pick("val_roc_auc"),
                "threshold": self.threshold, "device": str(self.device)}

    def predict(self, image):
        from models.vit.runtime import predict_image
        probability = predict_image(self.model, image, self.metadata, self.device)
        return {"prediction": "FAKE" if probability >= self.threshold else "REAL",
                "probability_fake": probability, "probability_real": 1.0 - probability,
                "threshold": self.threshold}

    def predict_path(self, path):
        with Image.open(path) as image:
            image.load()
            return self.predict(image)


def select_samples(manifest, data_root, per_source=8, seed=42, sealed_evaluation_complete=False):
    """Real dataset images with known labels, equal numbers per source.

    Before the sealed evaluation, validation images are used: they already guided checkpoint
    selection, so a demo on them cannot leak the sealed test sets (and their accuracy is slightly
    optimistic). Once the sealed evaluation is complete and frozen, never seen primary test images
    and the held out Stable Diffusion fakes are used instead.
    """
    frame = pd.read_csv(manifest)
    if sealed_evaluation_complete:
        plan = [("test", "FFHQ"), ("test", "CelebA"), ("test", "StyleGAN"), ("cross_gen", "StableDiffusion")]
    else:
        plan = [("val", "FFHQ"), ("val", "CelebA"), ("val", "StyleGAN")]
    samples = []
    for split, source in plan:
        pool = frame[(frame.split == split) & (frame.source == source)].sort_values("filepath")
        for row in pool.sample(n=min(per_source, len(pool)), random_state=seed).itertuples():
            samples.append({"path": str(Path(data_root) / row.filepath), "filepath": row.filepath,
                            "split": split, "source": source, "truth": "FAKE" if int(row.label) == 1 else "REAL"})
    if not samples:
        raise ValueError("No samples found in the manifest.")
    return samples


def folder_samples(folder, limit=200):
    """Your own images. The truth comes from a parent folder named real or fake; otherwise it is unknown."""
    folder = Path(folder)
    paths = sorted(path for path in folder.rglob("*") if path.is_file() and path.suffix.lower() in EXTENSIONS)
    samples = []
    for path in paths[:limit]:
        parents = {part.lower() for part in path.relative_to(folder).parts[:-1]}
        truth = None if {"real", "fake"} <= parents else "FAKE" if "fake" in parents else "REAL" if "real" in parents else None
        samples.append({"path": str(path), "filepath": str(path.relative_to(folder)), "split": "own",
                        "source": path.parent.name if path.parent != folder else folder.name, "truth": truth})
    return samples


SPLIT_NAMES = {"val": "validation", "test": "sealed test", "cross_gen": "held out Stable Diffusion", "own": "your images"}


def header_markdown(detector):
    s = detector.status
    progress = ("training complete" if s["training_complete"]
                else f"TRAINING INCOMPLETE: {s['epochs_recorded']} of {s['planned_epochs']} epochs recorded")
    quality = ""
    if s["val_accuracy_at_best"] is not None:
        quality = f" Validation accuracy at that epoch: {s['val_accuracy_at_best']:.4f}"
        if s["val_roc_auc_at_best"] is not None:
            quality += f", ROC AUC: {s['val_roc_auc_at_best']:.4f}"
        quality += "."
    return (f"# Real or fake face? ViT-B/16 detector\n"
            f"Run `{s['run_name']}`, best checkpoint from epoch {s['best_epoch']} ({progress}).{quality} "
            f"Verdict is **FAKE** when P(fake) ≥ {s['threshold']:.2f}. Device: {s['device']}.\n\n"
            "Trained on face crops: FFHQ and CelebA (real) against StyleGAN (fake). Stable Diffusion faces "
            "were held out to measure generalization. Photos that are not face crops like these, heavily edited "
            "images, or other generators are outside what this model has evidence for, so treat its score as a "
            "signal, not proof.")


def build_demo(detector, examples=None):
    import gradio as gr

    def classify(image):
        if image is None:
            raise gr.Error("Upload an image first.")
        result = detector.predict(image)
        headline = "FAKE" if result["prediction"] == "FAKE" else "REAL (not fake)"
        verdict = (f"## Verdict: {headline}\n"
                   f"P(fake) = {result['probability_fake']:.4f}, threshold {result['threshold']:.2f}")
        return {"FAKE": result["probability_fake"], "REAL": result["probability_real"]}, verdict, result

    def classify_batch(files):
        if not files:
            raise gr.Error("Upload one or more images.")
        rows = []
        for item in files:
            path = Path(item if isinstance(item, str) else item.name)
            try:
                result = detector.predict_path(path)
                rows.append([path.name, round(result["probability_fake"], 6), result["prediction"]])
            except Exception as error:  # one unreadable file must not hide the others
                rows.append([path.name, None, f"ERROR: {error}"])
        return pd.DataFrame(rows, columns=["file", "p_fake", "verdict"])

    with gr.Blocks(title="Real or fake face? ViT-B/16") as demo:
        gr.Markdown(header_markdown(detector))
        with gr.Tab("Single image"):
            with gr.Row():
                with gr.Column():
                    image = gr.Image(type="pil", image_mode="RGB", label="Face image")
                    check = gr.Button("Check image", variant="primary")
                with gr.Column():
                    verdict = gr.Markdown()
                    label = gr.Label(num_top_classes=2, label="Model probability")
                    details = gr.JSON(label="Raw output")
            if examples:
                gr.Examples(examples=[[sample["path"]] for sample in examples], inputs=image,
                            example_labels=[f"{sample['truth'] or 'unlabelled'} · {sample['source']}" for sample in examples],
                            label="Example images ("
                            + ", ".join(dict.fromkeys(SPLIT_NAMES[sample["split"]] for sample in examples)) + ")",
                            examples_per_page=12)
            check.click(classify, inputs=image, outputs=[label, verdict, details], api_name="predict")
        with gr.Tab("Batch check"):
            files = gr.File(file_count="multiple", file_types=["image"], type="filepath", label="Images")
            run_all = gr.Button("Check all", variant="primary")
            table = gr.Dataframe(headers=["file", "p_fake", "verdict"], label="Results")
            run_all.click(classify_batch, inputs=files, outputs=table, api_name="predict_batch")
    return demo


def self_test(url, detector, samples, tolerance=1e-5):
    """Send every sample through the running Gradio app over HTTP and compare with the model called directly.

    Returns a per image DataFrame and a summary. The pipeline check (upload, Gradio preprocessing,
    model, label) must match exactly; accuracy on these images is measured, never assumed.
    Samples whose truth is None (unlabelled own images) are predicted but not scored.
    """
    from gradio_client import Client, handle_file
    client = Client(url, verbose=False)
    rows = []
    for sample in samples:
        direct = detector.predict_path(sample["path"])
        label, _, details = client.predict(handle_file(sample["path"]), api_name="/predict")
        api_probability = float(details["probability_fake"])
        truth = sample["truth"]
        rows.append({"filepath": sample["filepath"], "path": sample["path"], "split": sample["split"],
                     "source": sample["source"], "truth": truth or "unknown", "prediction": direct["prediction"],
                     "p_fake": direct["probability_fake"],
                     "correct": (direct["prediction"] == truth) if truth else None,
                     "app_prediction": details["prediction"], "app_top_label": label["label"],
                     "app_p_fake": api_probability,
                     "app_matches_model": (abs(api_probability - direct["probability_fake"]) <= tolerance
                                           and details["prediction"] == direct["prediction"] == label["label"])})
    results = pd.DataFrame(rows)
    batch = client.predict([handle_file(sample["path"]) for sample in samples], api_name="/predict_batch")
    batch_frame = pd.DataFrame(batch["data"], columns=batch["headers"])
    batch_ok = (len(batch_frame) == len(results)
                and list(batch_frame.verdict) == list(results.prediction)
                and ((batch_frame.p_fake.astype(float) - results.p_fake).abs() <= tolerance).all())
    scored = results[results.correct.notna()]
    accuracy_of = lambda truth: (float(scored[scored.truth == truth].correct.astype(bool).mean())
                                 if (scored.truth == truth).any() else None)
    summary = {"images": len(results), "labelled_images": len(scored),
               "real_images": int((scored.truth == "REAL").sum()), "fake_images": int((scored.truth == "FAKE").sum()),
               "accuracy_on_real_images": accuracy_of("REAL"), "accuracy_on_fake_images": accuracy_of("FAKE"),
               "accuracy": float(scored.correct.astype(bool).mean()) if len(scored) else None,
               "accuracy_by_source": (scored.groupby("source").correct.apply(lambda c: round(float(c.astype(bool).mean()), 4)).to_dict()
                                      if len(scored) else {}),
               "predicted_fake": int((results.prediction == "FAKE").sum()),
               "predicted_real": int((results.prediction == "REAL").sum()),
               "single_image_endpoint_matches_model": bool(results.app_matches_model.all()),
               "batch_endpoint_matches_model": bool(batch_ok)}
    return results, summary


def plot_self_test(results, output, columns=8, title="Gradio self test (green = correct, red = wrong, grey = no label)"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rows = -(-len(results) // columns)
    fig, axes = plt.subplots(rows, columns, figsize=(2.1 * columns, 2.5 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, row in zip(axes.flat, results.itertuples()):
        with Image.open(row.path) as image:
            axis.imshow(image.convert("RGB"))
        color = "#555555" if row.correct is None or pd.isna(row.correct) else "#1a7f37" if row.correct else "#c62828"
        axis.set_title(f"true {row.truth} ({row.source})\npred {row.prediction} p={row.p_fake:.3f}", fontsize=7, color=color)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(output, dpi=140, bbox_inches="tight")
    plt.close(fig)
    return output


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--code-dir", required=True, help="folder containing the models package")
    parser.add_argument("--share", action="store_true", help="create a temporary public gradio.live link")
    parser.add_argument("--host", default=None, help="for example 0.0.0.0 to serve on your network or a server")
    parser.add_argument("--port", type=int, default=None)
    args = parser.parse_args()
    sys.path.insert(0, str(Path(args.code_dir).resolve()))
    build_demo(Detector(args.run_dir)).launch(share=args.share, server_name=args.host, server_port=args.port)


if __name__ == "__main__":
    main()

In [ ]:
def start_app_and_self_test():
    global GRADIO_DEMO
    import gradio
    if str(GRADIO_DIR) not in sys.path:
        sys.path.insert(0, str(GRADIO_DIR))
    import wish_gradio_app as app
    app = importlib.reload(app)  # pick up edits to the app cell on rerun
    if not (RUN_DIR / "checkpoints/best_model.pt").is_file():
        print("No trained checkpoint yet: the app starts once at least one training epoch has completed.")
        return None
    detector = app.Detector(RUN_DIR)
    status = OUTPUT_ROOT / "sealed_evaluation/status.json"
    sealed = status.is_file() and json.loads(status.read_text()).get("status") == "complete"
    samples = app.select_samples(AUDIT_DIR / "manifest.csv", DATA_ROOT, per_source=GRADIO_SELF_TEST_PER_SOURCE,
                                 seed=SEED, sealed_evaluation_complete=sealed)
    if OWN_TEST_IMAGES:
        own = app.folder_samples(OWN_TEST_IMAGES)
        labelled = sum(sample["truth"] is not None for sample in own)
        print(f"Adding {len(own)} of your own images ({labelled} labelled by a real/ or fake/ folder) from {OWN_TEST_IMAGES}.")
        samples += own
    # Copies of the example images live next to the app, so it can also be run outside Kaggle.
    examples_dir = GRADIO_DIR / "examples"
    shutil.rmtree(examples_dir, ignore_errors=True)
    examples_dir.mkdir()
    examples = []
    for number, sample in enumerate(samples):
        name = Path(sample["filepath"]).name.replace(" ", "_")
        target = examples_dir / f"{number:03d}_{(sample['truth'] or 'unlabelled').lower()}_{sample['source']}_{name}"
        shutil.copy2(sample["path"], target)
        examples.append(dict(sample, path=str(target)))

    if globals().get("GRADIO_DEMO") is not None:
        GRADIO_DEMO.close()  # a rerun replaces the previous app instead of occupying another port
        GRADIO_DEMO = None
    share = KAGGLE_RUN_TYPE != "Batch"  # Kaggle exposes no ports, so an interactive user needs the public link
    GRADIO_DEMO = app.build_demo(detector, examples)
    try:
        GRADIO_DEMO.launch(share=share, prevent_thread_lock=True, quiet=True, allowed_paths=[str(examples_dir)])
    except Exception as error:
        if not share:
            raise
        print(f"Public link unavailable ({error}); serving locally only.")
        GRADIO_DEMO.close()
        GRADIO_DEMO = app.build_demo(detector, examples)
        GRADIO_DEMO.launch(share=False, prevent_thread_lock=True, quiet=True, allowed_paths=[str(examples_dir)])

    results, summary = app.self_test(GRADIO_DEMO.local_url, detector, samples)
    summary.update({"splits_used": sorted({sample["split"] for sample in samples}),
                    "sealed_evaluation_complete": sealed, "run": detector.status, "gradio_version": gradio.__version__})
    results.to_csv(GRADIO_DIR / "self_test_results.csv", index=False)
    (GRADIO_DIR / "self_test_summary.json").write_text(json.dumps(summary, indent=2))
    app.plot_self_test(results, GRADIO_DIR / "self_test_grid.png")

    display(results[["source", "truth", "prediction", "p_fake", "correct", "app_matches_model"]])
    print(f"Self test through the running app: {summary['images']} images, of them {summary['real_images']} "
          f"known real and {summary['fake_images']} known fake, from: {', '.join(summary['splits_used'])}.")
    percent = lambda value: "n/a" if value is None else f"{value:.1%}"
    print(f"Correct verdicts: {percent(summary['accuracy'])} overall | real images "
          f"{percent(summary['accuracy_on_real_images'])} | fake images {percent(summary['accuracy_on_fake_images'])}")
    print("By source:", summary["accuracy_by_source"])
    print("App output equals direct model output: single image endpoint",
          summary["single_image_endpoint_matches_model"], "| batch endpoint", summary["batch_endpoint_matches_model"])
    display(DisplayImage(filename=str(GRADIO_DIR / "self_test_grid.png")))
    if not detector.status["training_complete"]:
        print("NOTE: training is not complete yet; this tests the best checkpoint so far, not the final model.")
    if summary["labelled_images"] and (summary["predicted_fake"] == 0 or summary["predicted_real"] == 0):
        print("WARNING: every image received the same verdict. Inspect the training curves before trusting the model.")
    if not (summary["single_image_endpoint_matches_model"] and summary["batch_endpoint_matches_model"]):
        raise RuntimeError("The app's answers differ from the model's. See self_test_results.csv.")

    if KAGGLE_RUN_TYPE == "Batch":
        GRADIO_DEMO.close()
        print("Saved run: app tested and shut down. Open an interactive session to use it (see the section notes).")
    else:
        if GRADIO_DEMO.share_url:
            print("App running. Open:", GRADIO_DEMO.share_url)
        else:
            print("App running only at", GRADIO_DEMO.local_url, "which Kaggle does not expose to your browser. "
                  "Turn Internet on for a public link, or run the app elsewhere (next section).")
        print("It stays up while this session lives; GRADIO_DEMO.close() stops it.")
    return summary


from IPython.display import Image as DisplayImage, display
if RUN_GRADIO_APP and gradio_version():
    GRADIO_SUMMARY = gradio_step("Gradio app and self test", start_app_and_self_test)
elif RUN_GRADIO_APP:
    print("Gradio is not available; see the installation cell above.")

## Using the app after this run
**On Kaggle:** open an interactive session with this notebook's output attached and run the settings,
runtime, dataset and Gradio cells (nothing is retrained). The cell prints a public `gradio.live` link.

**Anywhere else (a laptop or a Hugging Face Space):** download the output folder `wish_vit_ddp_v4`, then
```
pip install torch torchvision timm==1.0.30 "gradio>=5,<7" pandas pillow scikit-learn matplotlib
python gradio_app/wish_gradio_app.py --run-dir runs/vit_wish_s42_ddp2_v1 --code-dir code
```
Add `--share` for a temporary public link. The app needs only `code/models`, `gradio_app/wish_gradio_app.py`,
and from `runs/<RUN_NAME>`: `run_config_used.json`, `training_history.csv` and `checkpoints/best_model.pt`
(about 330 MB). The large `last.pt` resume state is not needed. It runs on CPU as well, at a fraction of a second
per image.

**Scope:** the model has evidence for face crops like Wish's: FFHQ and CelebA real faces against StyleGAN
fakes, with Stable Diffusion held out to measure generalization. Uncropped photos, heavily edited or
recompressed images, and other generators are outside that evidence, so treat the score as a signal, not proof.